# 4. Dashboard visuals (pipeline step 9)

This notebook runs **pipeline step 9** (`9_dashboard_visuals`). It **prebuilds all dashboard visualization artifacts** (**Scenario Analysis** (scenario-based SHAP+FFA combined), BupaR, DTW, FP-Growth, **Cohort PGx** for the PGx Cohort tab, and **PGx Card** for the patient card tab) on **EC2** and **saves them to the final local destination** that notebook 5 Step 6 syncs to S3 (`10_risk_dashboard/visualizations/{bupar,dtw,fpgrowth,cohort_pgx,causal}/`). The dashboard loads these prebuilt assets **static-first** from S3 using the **manifest** (`10_risk_dashboard/visualizations/dashboard_visual_objects.json`) as the single source of truth for tab → S3 paths and static files; **JSON-first** for visuals (e.g. DTW: chart_data, sequence_heatmap, trajectory_overview_plot from static, then PNG/HTML fallbacks; API only when static fails). The API returns URLs to prebuilt S3 assets; no computation at request time. Visuals are **SHAP/FFA-driven**: model data and feature lists come from Step 3b / 7 / 8 so process mining and itemset mining use only important features.

**Flow:** Run after [3_model_train_shap_ffa.ipynb](3_model_train_shap_ffa.ipynb). Then run [5_build_and_deploy.ipynb](5_build_and_deploy.ipynb) once to build and deploy.

**Prerequisites (all contained in this notebook):**
- **Primary**: Step 3b feature importance (`3b_feature_importance_eda/outputs/{cohort}/{age_band}/` or `3a_feature_importance/...`) or combined importance (`10_risk_dashboard/outputs/{cohort}/{age_band}/combined_importance.csv`).
- **No need to re-run an earlier notebook**: If neither exists, this notebook runs the **combine SHAP+FFA** step (Steps 7 and 8) to generate `combined_importance.csv` in `10_risk_dashboard/outputs/`, then proceeds with allowed_codes, BupaR, DTW, FP-Growth, and Cohort PGx. You only need 7_shap_analysis and 8_ffa_analysis outputs to exist.

## Steps

1. **Setup** – Resolve paths (scripts in `9_dashboard_visuals/`; outputs under `10_risk_dashboard/visualizations/{scenario,bupar,dtw,fpgrowth,cohort_pgx}/`).
2. **Feature importance heatmaps** – Aggregated and combined heatmaps for the dashboard **Feature Importance** tab; saved to `3a_feature_importance/{cohort}/plots/{cohort}_aggregated_fi_heatmap.png` and `3a_feature_importance/plots/combined_cohorts_feature_importance_heatmap.png`. Notebook 5 (deploy) expects these paths and syncs them to S3.
3. **BupaR** – Process mining sequences and plots (SHAP/FFA allowed codes when available); **saved locally** to `10_risk_dashboard/visualizations/bupar/{cohort}/{age_band_fname}/plots/` (notebook 5 Step 6 syncs to S3 `visualizations/bupar/{cohort}/{age_band}/plots/`).
4. **DTW** – Trajectory features and plots **based on SHAP/FFA important codes** (same as BupaR/FP-Growth); **drug-only** for both cohorts. **Saved locally** to `10_risk_dashboard/visualizations/dtw/{cohort}/{age_band_fname}/`: `chart_data.json`, `sequence_heatmap.json`, `plots/trajectory_overview_plot.json`, `plots/dtw_trajectory_analysis_{base}.png`, `plots/dtw_sample_trajectories_{base}.png`, `plots/dtw_trajectory_cluster_1d/3d_{base}.html`. Notebook 5 Step 6 syncs to S3 `visualizations/dtw/{cohort}/{age_band}/` (age_band with **hyphen** on S3). **No empty artifacts:** when a plot doesn't produce data, the pipeline writes JSON with `message`, `empty: true`, and `metrics` (why). The **manifest** (`dashboard_visual_objects.json`) lists all DTW objects and S3 paths; the dashboard uses **JSON-first** loading (chart_data, sequence_heatmap, trajectory_overview_plot from static first, then PNG/HTML fallbacks, API only when static fails). DTW tab includes **Routine vs No Routine (Outcomes)** (admin ICD codes identify routine appointments) and **High-Risk vs Low-Risk Trajectories**. Full pipeline (2016–2019). **Extreme-density** (optional): same routine vs no routine for high-utilizer subgroups.

5. **FP-Growth** – Itemsets, rules, **Plotly network HTML**, and PNGs; **saved locally** to `10_risk_dashboard/visualizations/fpgrowth/{cohort}/{age_band_fname}/plots/` and `.../data/` (notebook 5 Step 6 syncs to S3 `visualizations/fpgrowth/{cohort}/{age_band}/`). The dashboard loads the **network plot by cohort** from these URLs.

**Scenario (Scenario Analysis tab)** – Combined SHAP+FFA produces `dashboard_data.json` per cohort/age_band (via `combine_shap_ffa_results.py`); saved under `10_risk_dashboard/visualizations/scenario/{cohort}/{age_band_fname}/`. Run **Upload Scenario dashboard JSON to S3** (cell below) to push to the dashboard bucket as `visualizations/scenario/{cohort}/{age_band}/scenario_data.json`. The Scenario Analysis tab loads from S3; notebook 5 Step 6 also runs this during deploy.

Idempotent. Run from repo root. Prerequisites: `4_model_data`, `7_shap_analysis`, `8_ffa_analysis`; R and bupaR for BupaR. Then run notebook 5 to deploy (syncs all visuals including Scenario, Cohort PGx, and others to S3).


6. **Cohort PGx (PGx Cohort tab)** – Three-step pipeline per cohort/age_band (full-cohort + per density bin):
   - **Step 1**: Fetch PharmGKB VIP reports (`fetch_vip_reports.py`) for top-N SHAP/FFA genes.
   - **Step 1.5**: Fetch PubMed citations (`fetch_pubmed_citations.py`) following the `lit_review` `search_pubmed_all` pattern — last 5 years, pharmacogenomics MeSH + cohort-context (opioid / emergency department) queries, XML efetch for PMC IDs, BioC JSON full-text URLs. Outputs `pubmed_citations.json` alongside `network_topology.html`; rendered in dashboard as "Supporting Literature" accordion.
   - **Step 2**: Build interactive gene–drug–phenotype network topology (`build_network_topology.py`).
   - **Saved locally** to `10_risk_dashboard/visualizations/cohort_pgx/networks/{cohort}/{age_band_fname}/[density/{bin}/]`. Notebook 5 Step 6 syncs to S3 `visualizations/cohort_pgx/networks/{cohort}/{age_band}/`. The **PGx Cohort** dashboard tab calls GET /visualizations/cohort-pgx and displays the network iframe + citations section.

7. **PGx Card (optional)** – Prepares CPIC gene–drug data, PharmGKB VIP JSON, and QR codes for the **PGx Card** dashboard tab. Lambda `POST /pgx/card` uses `pgx-patient-card/data/` (e.g. `cpic_gene-drug_pairs.xlsx`, `pharmgkb_vip_genes.json`, `pgx_database`); notebook 5 packages these into the Lambda image.

8. **Model performance metrics and cohort metadata** – Prebuilt via `generate_metrics.py` and `generate_metadata.py` (no recomputation). Deploy (5_build_and_deploy) uploads to the dashboard bucket: `metadata/model_performance_metrics.json` (Documentation tab) and `metadata/opioid_ed.json`, `metadata/non_opioid_ed.json` (dropdowns). Frontend loads these same-origin; Lambda GET /metrics and GET /metadata are fallbacks.

9. **API** – Returns URLs to prebuilt S3 assets only (no server-side computation for visuals). Lambda GET /visualizations/scenario serves the Scenario Analysis tab; GET /visualizations/cohort-pgx serves the PGx Cohort tab; Lambda POST /pgx/card serves the PGx Card tab.

In [1]:
# Setup: paths (outputs under 10_risk_dashboard/visualizations/)
# Notebook 4 builds visuals to LOCAL only. Notebook 5 Step 6 is the single place that syncs to S3 (idempotent).
import sys
import os
import subprocess
from pathlib import Path

os.environ["SKIP_DASHBOARD_S3_UPLOAD"] = "1"  # Write local only; notebook 5 Step 6 syncs to S3

REPO_ROOT = Path.cwd()
if (REPO_ROOT / "py_helpers").exists():
    pass  # already repo root
else:
    for p in REPO_ROOT.parents:
        if (p / "py_helpers").exists():
            REPO_ROOT = p
            break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from py_helpers.env_utils import get_workflow_python_bin
PYTHON_BIN = get_workflow_python_bin()  # EC2: jupyter-env; local: sys.executable

from py_helpers.env_utils import get_data_root, get_model_data_root

# Data root (with fallback to local nvme or Windows pgx_data)
DATA_ROOT = get_data_root()
MODEL_DATA_ROOT = get_model_data_root()
S3_BUCKET = os.environ.get("PGX_S3_BUCKET", "pgxdatalake")



# Minimal bootstrap if you skipped Config
from py_helpers.constants import REQUIRED_COHORTS, DASHBOARD_VISUAL_AGE_BANDS

FORCE_RERUN = True
PARALLEL_WORKERS = 12

# Full dashboard scope (both cohorts, 13-24 … 85-114):
combinations = [
    (c, ab) for c, bands in REQUIRED_COHORTS.items()
    for ab in bands if ab in DASHBOARD_VISUAL_AGE_BANDS
]

print(f"DTW will run {len(combinations)} jobs: {combinations[:3]}...")

# Creation code (step 9); outputs go to 10_risk_dashboard/visualizations
STEP9_ROOT = REPO_ROOT / "9_dashboard_visuals"
VISUAL_ROOT = REPO_ROOT / "10_risk_dashboard" / "visualizations"
BUPAR_VISUALS_SCRIPT = STEP9_ROOT / "bupar" / "create_bupar_visuals.py"
DTW_TRAJECTORIES_SCRIPT = STEP9_ROOT / "dtw" / "create_dtw_trajectories.py"
DTW_FEATURES_SCRIPT = STEP9_ROOT / "dtw" / "create_dtw_features.py"
DTW_VISUALS_SCRIPT = STEP9_ROOT / "dtw" / "create_dtw_visuals.py"
FPGROWTH_VISUALS_SCRIPT = STEP9_ROOT / "fpgrowth" / "create_fpgrowth_visuals.py"

print(f"Repo root: {REPO_ROOT}")
print(f"Data root (NVMe/local): {DATA_ROOT}")
print(f"Model data root: {MODEL_DATA_ROOT}")
print(f"S3 bucket: {S3_BUCKET}")
print(f"Step 9 (scripts): {STEP9_ROOT}")
print(f"Outputs: {VISUAL_ROOT}")
print(f"Plots (PNG/HTML) appear under: {VISUAL_ROOT}/bupar/<cohort>/<age_band>/plots/ (and dtw/, fpgrowth/, cohort_pgx/networks/, causal/) — run the cells below to generate them.")

Repo root: /home/pgx3874/pgx-analysis
Data root (NVMe/local): /mnt/nvme
Model data root: /mnt/nvme/4_model_data
S3 bucket: pgxdatalake
Step 9 (scripts): /home/pgx3874/pgx-analysis/9_dashboard_visuals
Outputs: /home/pgx3874/pgx-analysis/10_risk_dashboard/visualizations
Plots (PNG/HTML) appear under: /home/pgx3874/pgx-analysis/10_risk_dashboard/visualizations/bupar/<cohort>/<age_band>/plots/ (and dtw/, fpgrowth/, cohort_pgx/networks/, causal/) — run the cells below to generate them.


## Config: cohorts and age bands

Defaults match **run_dashboard_visuals.py**: all cohorts and all age bands (from REQUIRED_COHORTS), one worker per (cohort, age_band) combo (capped by CPU), no dry run. Leave `COHORTS_TO_RUN` and `AGE_BANDS_TO_RUN` empty for full pipeline; set either to limit scope.

### Upload Scenario dashboard JSON to S3 (Scenario Analysis tab)

Run `upload_scenario_outputs_to_s3.py` to upload **scenario visualizations** from `10_risk_dashboard/visualizations/scenario/{cohort}/{age_band_fname}/dashboard_data.json` (produced by `combine_shap_ffa_results.py`) to the dashboard bucket as `visualizations/scenario/{cohort}/{age_band}/scenario_data.json`. The Scenario Analysis tab loads from S3; notebook 5 Step 6 also runs this during deploy. Running here lets the tab have data after building visuals in this notebook.

In [3]:
# Upload scenario dashboard JSON to S3 (Scenario Analysis tab)
# Script reads 10_risk_dashboard/visualizations/scenario/{cohort}/{age_band_fname}/dashboard_data.json
upload_scenario_script = REPO_ROOT / "10_risk_dashboard" / "data_preparation" / "upload_scenario_outputs_to_s3.py"
if upload_causal_script.exists():
    r = subprocess.run(
        [str(PYTHON_BIN), str(upload_scenario_script)],
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    if r.returncode == 0:
        print(r.stdout if r.stdout else "✓ Scenario dashboard JSON uploaded to S3 (visualizations/scenario/{cohort}/{age_band}/scenario_data.json)")
    else:
        print("⚠ Upload scenario outputs to S3:", r.stderr or r.stdout or "non-zero exit")
else:
    print("  upload_scenario_outputs_to_s3.py not found; Step 6 in notebook 5 will upload scenario outputs during deploy.")

  ✓ Scenario data (bin=low): opioid_ed/0-12/low -> s3://jerome-dixon.io/vcu/pgx-risk-calculator/visualizations/scenario/opioid_ed/0-12/low/scenario_data.json
  ✓ Scenario data (bin=medium): opioid_ed/0-12/medium -> s3://jerome-dixon.io/vcu/pgx-risk-calculator/visualizations/scenario/opioid_ed/0-12/medium/scenario_data.json
  ✓ Scenario data (bin=high): opioid_ed/0-12/high -> s3://jerome-dixon.io/vcu/pgx-risk-calculator/visualizations/scenario/opioid_ed/0-12/high/scenario_data.json
  ✓ Scenario data (bin=low): opioid_ed/13-24/low -> s3://jerome-dixon.io/vcu/pgx-risk-calculator/visualizations/scenario/opioid_ed/13-24/low/scenario_data.json
  ✓ Scenario data (bin=medium): opioid_ed/13-24/medium -> s3://jerome-dixon.io/vcu/pgx-risk-calculator/visualizations/scenario/opioid_ed/13-24/medium/scenario_data.json
  ✓ Scenario data (bin=high): opioid_ed/13-24/high -> s3://jerome-dixon.io/vcu/pgx-risk-calculator/visualizations/scenario/opioid_ed/13-24/high/scenario_data.json
  ✓ Scenario data (bin

### Sync FFA raw outputs to S3 + generate scenario manuscript summary

Upload `8_ffa_analysis/outputs/` (ffa_causal_factors.csv, axp_explanations.parquet,
feature_importance_axp.parquet) to `s3://pgxdatalake/gold/ffa_analysis/` and write
`gold/ffa_analysis/{cohort}/{age_band}/{bin}/ffa_manuscript_summary.json` with top
drug features, IR scores, and rule-profile counts needed for CH_4 manuscript placeholders.

In [ ]:

# Sync FFA raw outputs to S3 + generate scenario manuscript summary JSON
# Uploads: ffa_causal_factors.csv, axp_explanations.parquet, feature_importance_axp.parquet
# Destination: s3://pgxdatalake/gold/ffa_analysis/{cohort}/{age_band}/bin_models/{bin}/{model}/
# Also writes scenario-based summary: gold/ffa_analysis/{cohort}/{age_band}/{bin}/ffa_manuscript_summary.json

import boto3
import json
import pandas as pd
from pathlib import Path

S3_FFA_PREFIX = "gold/ffa_analysis"
FFA_OUTPUTS = REPO_ROOT / "8_ffa_analysis" / "outputs"
s3_client = boto3.client("s3", region_name="us-east-1")

UPLOAD_EXTENSIONS = {".csv", ".parquet", ".json"}
MANUSCRIPT_FILES = {"ffa_causal_factors.csv", "axp_explanations.parquet", "feature_importance_axp.parquet"}

uploaded = 0
skipped = 0
errors = 0
summaries_written = 0

# ── Upload raw FFA outputs ────────────────────────────────────────────────────
print("=" * 70)
print("Uploading FFA raw outputs to S3 ...")
print("=" * 70)

for fpath in sorted(FFA_OUTPUTS.rglob("*")):
    if not fpath.is_file():
        continue
    if fpath.suffix not in UPLOAD_EXTENSIONS:
        continue
    if fpath.name not in MANUSCRIPT_FILES and fpath.suffix not in {".json"}:
        continue  # only key files + any JSON summaries

    rel = fpath.relative_to(FFA_OUTPUTS)          # e.g. non_opioid_ed/65_74/bin_models/low/xgboost/ffa_causal_factors.csv
    # Normalise age_band underscores → hyphens in S3 key
    parts = list(rel.parts)
    if len(parts) >= 2:
        parts[1] = parts[1].replace("_", "-")     # age_band dir
    s3_key = S3_FFA_PREFIX + "/" + "/".join(parts)

    try:
        s3_client.upload_file(str(fpath), S3_BUCKET, s3_key)
        print(f"  ✓  {s3_key}")
        uploaded += 1
    except Exception as exc:
        print(f"  ✗  {s3_key}  — {exc}")
        errors += 1

print(f"\nUpload complete: {uploaded} uploaded, {errors} errors.\n")

# ── Generate per-bin manuscript summary JSON ─────────────────────────────────
print("=" * 70)
print("Generating ffa_manuscript_summary.json per cohort / age_band / bin ...")
print("=" * 70)

from py_helpers.event_density_utils import DENSITY_BINS

for cohort_dir in sorted(FFA_OUTPUTS.iterdir()):
    if not cohort_dir.is_dir():
        continue
    cohort_name = cohort_dir.name

    for ab_dir in sorted(cohort_dir.iterdir()):
        if not ab_dir.is_dir():
            continue
        age_band_fname = ab_dir.name                          # underscore form
        age_band = age_band_fname.replace("_", "-")

        bin_models_dir = ab_dir / "bin_models"
        bins_to_scan = sorted(bin_models_dir.iterdir()) if bin_models_dir.exists() else [ab_dir]

        for bin_dir in bins_to_scan:
            bin_name = bin_dir.name if bin_models_dir.exists() else "cohort"
            summary = {
                "cohort": cohort_name,
                "age_band": age_band,
                "bin": bin_name,
                "n_causal_features": 0,
                "top_features": [],
                "n_ffa_rules": 0,
                "top_ffa_rules": [],
            }

            # ── ffa_causal_factors.csv (top scenario-supported SHAP/FFA features) ──
            causal_csv = bin_dir / "ffa_causal_factors.csv"
            if not causal_csv.exists():
                # also check directly under bin_dir (cohort-level)
                causal_csv = ab_dir / "ffa_causal_factors.csv"

            if causal_csv.exists():
                try:
                    cf = pd.read_csv(causal_csv)
                    summary["n_causal_features"] = len(cf)
                    # Detect IR / importance column
                    ir_col = next((c for c in ("intervention_rate", "ir", "IR", "importance",
                                               "ffa_importance", "mean_abs_shap") if c in cf.columns), None)
                    feat_col = next((c for c in ("feature", "feature_name", "code", "item") if c in cf.columns), None)
                    if feat_col and ir_col:
                        top = cf.nlargest(15, ir_col)[[feat_col, ir_col]].copy()
                        top.columns = ["feature", "ir_score"]
                        summary["top_features"] = top.to_dict(orient="records")
                except Exception as e:
                    summary["causal_csv_error"] = str(e)

            # ── axp_explanations.parquet (rule-profile support; not causal drug-drug synergy) ──
            for model_sub in ("xgboost", "catboost"):
                axp_path = bin_dir / model_sub / "axp_explanations.parquet"
                if not axp_path.exists():
                    axp_path = bin_dir / model_sub / "axp_explanations.csv"
                if not axp_path.exists():
                    continue
                try:
                    axp = pd.read_parquet(axp_path) if axp_path.suffix == ".parquet" else pd.read_csv(axp_path)
                    summary["n_ffa_rules"] = len(axp)
                    # Extract top rule profiles by available score if present
                    ie_col = next((c for c in ("interaction_effect", "ie_score", "ie", "IE",
                                               "synergy_score", "probability_shift") if c in axp.columns), None)
                    rule_col = next((c for c in ("rule", "rule_str", "antecedent", "items") if c in axp.columns), None)
                    if ie_col and rule_col:
                        top_rules = axp.nlargest(10, ie_col)[[rule_col, ie_col]].copy()
                        top_rules.columns = ["rule", "ie_score"]
                        summary["top_ffa_rules"] = top_rules.to_dict(orient="records")
                    summary["model"] = model_sub
                    break
                except Exception as e:
                    summary[f"axp_error_{model_sub}"] = str(e)

            # ── Upload summary JSON ───────────────────────────────────────
            summary_key = f"{S3_FFA_PREFIX}/{cohort_name}/{age_band}/{bin_name}/ffa_manuscript_summary.json"
            try:
                s3_client.put_object(
                    Bucket=S3_BUCKET,
                    Key=summary_key,
                    Body=json.dumps(summary, indent=2, default=str).encode(),
                    ContentType="application/json",
                )
                print(f"  ✓  {cohort_name}/{age_band}/{bin_name}  "
                      f"features={summary['n_causal_features']}  rules={summary['n_ffa_rules']}")
                summaries_written += 1
            except Exception as exc:
                print(f"  ✗  {summary_key}  — {exc}")

print(f"\nSummaries written: {summaries_written}")
print(f"S3 prefix: s3://{S3_BUCKET}/{S3_FFA_PREFIX}/")
print("\nTo read in manuscript scripts:")
print(f"  aws s3 cp s3://{S3_BUCKET}/{S3_FFA_PREFIX}/non_opioid_ed/65-74/low/ffa_manuscript_summary.json -")


In [ ]:
# Prerequisite: n_event_bin_thresholds.json (written by Step 6 run_final_model.py in notebook 3).
# DTW, BupaR, FP-Growth, and risk inference load these thresholds to apply the same
# low/medium/high/extreme bin cuts as the trained model. If EC2 local files are missing,
# pull the deployed copies from S3 before failing.
import json

FINAL_MODEL_OUTPUTS = REPO_ROOT / "6_final_model" / "outputs"
MODEL_THRESHOLDS_S3_PREFIX = "gold/dashboard/models"

def _valid_threshold_json(path: Path) -> bool:
    try:
        data = json.loads(path.read_text())
        return all(k in data for k in ("p25", "p50", "p95"))
    except Exception:
        return False

try:
    import boto3
    _threshold_s3 = boto3.client("s3", region_name="us-east-1")
except Exception as exc:
    _threshold_s3 = None
    print(f"[WARN] boto3 unavailable; cannot pull missing n_event_bin_thresholds.json from S3: {exc}")

_thresh_missing = []
_thresh_downloaded = 0
for cohort_name, age_band in combinations:
    age_band_fname = age_band.replace("-", "_")
    thresh_path = FINAL_MODEL_OUTPUTS / cohort_name / age_band_fname / "n_event_bin_thresholds.json"
    if thresh_path.exists() and _valid_threshold_json(thresh_path):
        continue

    # Some Step 6/per-bin runs write the same thresholds only under bin_models/{bin}/.
    # Promote the first valid per-bin copy to the canonical cohort/age root path.
    for bin_name in ("low", "medium", "high", "extreme"):
        bin_thresh_path = FINAL_MODEL_OUTPUTS / cohort_name / age_band_fname / "bin_models" / bin_name / "n_event_bin_thresholds.json"
        if bin_thresh_path.exists() and _valid_threshold_json(bin_thresh_path):
            thresh_path.parent.mkdir(parents=True, exist_ok=True)
            thresh_path.write_text(bin_thresh_path.read_text())
            _thresh_downloaded += 1
            break
    if thresh_path.exists() and _valid_threshold_json(thresh_path):
        continue

    s3_key = f"{MODEL_THRESHOLDS_S3_PREFIX}/{cohort_name}/{age_band_fname}/n_event_bin_thresholds.json"
    if _threshold_s3 is not None:
        try:
            thresh_path.parent.mkdir(parents=True, exist_ok=True)
            obj = _threshold_s3.get_object(Bucket=S3_BUCKET, Key=s3_key)
            data = json.loads(obj["Body"].read().decode("utf-8"))
            if all(k in data for k in ("p25", "p50", "p95")):
                thresh_path.write_text(json.dumps(data, indent=2))
                _thresh_downloaded += 1
                continue
            _thresh_missing.append(f"{cohort_name}/{age_band} invalid S3 JSON: s3://{S3_BUCKET}/{s3_key}")
            continue
        except Exception as exc:
            _thresh_missing.append(f"{cohort_name}/{age_band} ({thresh_path}; S3 s3://{S3_BUCKET}/{s3_key}: {exc})")
            continue

    _thresh_missing.append(f"{cohort_name}/{age_band} ({thresh_path}; S3 not checked)")

if _thresh_missing:
    msg = (
        "n_event_bin_thresholds.json not found for some cohorts/age-bands locally or in S3.\n"
        "These files are written by Step 6 (run_final_model.py) and deployed with prepare_models.py.\n"
        "Run notebook 3, or sync/deploy model artifacts to s3://{bucket}/{prefix}/.\n"
        "Missing:\n" + "\n".join(f"  {m}" for m in _thresh_missing)
    ).format(bucket=S3_BUCKET, prefix=MODEL_THRESHOLDS_S3_PREFIX)
    raise RuntimeError(msg)
print(
    f"Prerequisite check passed: n_event_bin_thresholds.json present for all {len(combinations)} "
    f"cohort/age-band combinations ({_thresh_downloaded} downloaded from S3)."
)


## Feature importance heatmaps (dashboard Feature Importance tab)

Build aggregated and combined feature importance heatmaps from Step 3a outputs. **Saved locations** (used by notebook 5 and deploy sync):

- Per cohort: `3a_feature_importance/{cohort}/plots/{cohort}_aggregated_fi_heatmap.png`
- Combined: `3a_feature_importance/plots/combined_cohorts_feature_importance_heatmap.png`

Prerequisite: Step 3a aggregated CSVs at `3a_feature_importance/{cohort}/{cohort}_{age_band}_aggregated_feature_importance.csv`.

### Convert FI CSVs to JSON (cohort / model / age_band filters)

Run after the heatmaps cell. Scans `3a_feature_importance/outputs` for aggregated and per-model CSVs, writes:
- `feature_importance_index.json` — available cohorts, age_bands, models for dashboard dropdowns
- `{cohort}/plots/{cohort}_{model}_fi_heatmap.json` — heatmap when 2+ age bands (model = aggregated, catboost, xgboost, xgboost_rf)
- `{cohort}/plots/{cohort}_{model}_{age_band}_fi.json` — single-age feature list for bar chart

Deploy (notebook 5) uploads these to S3 so the Feature Importance tab can filter by model and age_band.

In [ ]:
# Build FI heatmaps for dashboard (saved where notebook 5 / deploy expect them)
from py_helpers.feature_importance_heatmap import create_aggregated_fi_heatmap, create_combined_cohorts_fi_heatmap

FI_OUTPUTS_BASE = REPO_ROOT / "3a_feature_importance" / "outputs"
# REQUIRED_COHORTS from Config cell: {cohort: [age_bands]}
paths_per_cohort = []
for cohort, age_bands in REQUIRED_COHORTS.items():
    p = create_aggregated_fi_heatmap(cohort, age_bands, FI_OUTPUTS_BASE, top_n=50)
    if p:
        paths_per_cohort.append(p)
        print(f"  ✓ {cohort}: {p}")
    else:
        print(f"  ✗ {cohort}: no aggregated CSVs found under {FI_OUTPUTS_BASE / cohort}")

combined_path = create_combined_cohorts_fi_heatmap(FI_OUTPUTS_BASE, REQUIRED_COHORTS, top_n=80)
if combined_path:
    print(f"  ✓ Combined: {combined_path}")
else:
    print("  ✗ Combined: no data (ensure at least one cohort has aggregated CSVs)")

print()
print("Save locations (must match notebook 5 requirement check and deploy sync):")
print(f"  Per cohort: {FI_OUTPUTS_BASE}/<cohort>/plots/<cohort>_aggregated_fi_heatmap.png")
print(f"  Combined:  {FI_OUTPUTS_BASE}/plots/combined_cohorts_feature_importance_heatmap.png")

# Copy FI heatmaps to 10_risk_dashboard/visualizations/feature_importance/ (same location as other dashboard visuals)
import shutil
FI_VIZ = REPO_ROOT / "10_risk_dashboard" / "visualizations" / "feature_importance"
FI_VIZ.mkdir(parents=True, exist_ok=True)
copied = 0
for cohort in REQUIRED_COHORTS:
    for base in (FI_OUTPUTS_BASE, REPO_ROOT / "3a_feature_importance"):
        src_plots = base / cohort / "plots"
        if not src_plots.exists():
            continue
        dest_dir = FI_VIZ / cohort
        dest_dir.mkdir(parents=True, exist_ok=True)
        for ext in ("png", "json"):
            src = src_plots / f"{cohort}_aggregated_fi_heatmap.{ext}"
            if src.exists():
                shutil.copy2(src, dest_dir / f"aggregated_fi_heatmap.{ext}")
                copied += 1
        for model in ("catboost", "xgboost", "xgboost_rf"):
            src_m = src_plots / f"{cohort}_{model}_fi_heatmap.json"
            if src_m.exists():
                shutil.copy2(src_m, dest_dir / f"{model}_fi_heatmap.json")
                copied += 1
        break
for base in (FI_OUTPUTS_BASE, REPO_ROOT / "3a_feature_importance"):
    if (base / "plots" / "combined_cohorts_feature_importance_heatmap.png").exists():
        shutil.copy2(base / "plots" / "combined_cohorts_feature_importance_heatmap.png", FI_VIZ / "combined_cohorts_feature_importance_heatmap.png")
        copied += 1
    combined_json = base / "combined" / "aggregated_fi_heatmap.json"
    if combined_json.exists():
        (FI_VIZ / "combined").mkdir(parents=True, exist_ok=True)
        shutil.copy2(combined_json, FI_VIZ / "combined" / "aggregated_fi_heatmap.json")
        copied += 1
    break
if copied:
    print(f"\n  Copied {copied} FI file(s) to {FI_VIZ} (notebook 5 / deploy sync from here).")

## Run BupaR process mining

Plots are **saved locally** to `10_risk_dashboard/visualizations/bupar/{cohort}/{age_band_fname}/plots/` (the final destination; notebook 5 Step 6 syncs to S3 `visualizations/bupar/{cohort}/{age_band}/plots/`). **BupaR features are not used for feature engineering** (same as DTW and FP-Growth); they are computed for dashboard visualization and analysis.

In [ ]:
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

FAIL_FAST = True
FORCE_RERUN = True
force_flag = ["--force"] if FORCE_RERUN else []

def run_bupar_one(cohort_name, age_band):
    r = subprocess.run(
        [str(PYTHON_BIN), str(BUPAR_VISUALS_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band] + force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    return (cohort_name, age_band, r.returncode, r.stdout, r.stderr)

with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as ex:
    futures = {ex.submit(run_bupar_one, c, ab): (c, ab) for c, ab in combinations}
    for fut in as_completed(futures):
        cohort_name, age_band, code, stdout, stderr = fut.result()
        print(f"  [BupaR] {cohort_name} / {age_band} -> exit {code}")
        if code != 0:
            ab_f = age_band.replace("-", "_")
            print(f"    create_bupar_visuals failed (exit {code}). Check 9_dashboard_visuals/logs/bupaR/bupar_{cohort_name}_{ab_f}.log if available")
            if stderr:
                print("    stderr:", (stderr[:1500] + "..." if len(stderr) > 1500 else stderr))
            if stdout:
                print("    stdout:", (stdout[:800] + "..." if len(stdout) > 800 else stdout))
            if FAIL_FAST:
                raise RuntimeError(f"BupaR failed: {cohort_name} / {age_band}")
print("BupaR done.")

## Run DTW trajectory analysis (visualization and analysis)

**DTW trajectories and alignment are for visualization and analysis** (not feature engineering). **DTW is drug-only for both cohorts** (opioid_ed and non_opioid_ed): only prescription (drug) events in the SHAP/FFA allowed set are included. **Routine vs no routine** uses administrative ICD codes (e.g. well visits, screenings) to identify routine appointments; `admin_icd_event_count` drives those charts.

This step:
1. **Extracts trajectories** from model_data filtered by SHAP/FFA important **drug** codes (same allowed-codes flow as BupaR/FP-Growth)
2. **Sequence alignment** – DTW distance computation to prototype trajectories to identify common patterns
3. **Creates visualizations** (trajectory cluster plots, routine vs no routine charts, sequence heatmaps)
4. **Saves locally** to `10_risk_dashboard/visualizations/dtw/{cohort}/{age_band_fname}/` (EC2 paths use **underscore** in age_band). Notebook 5 Step 6 syncs to S3 `visualizations/dtw/{cohort}/{age_band}/` (S3 uses **hyphen**).

**No empty artifacts.** When a plot doesn’t produce data, the pipeline **always** writes a JSON artifact with `message`, `empty: true`, `cohort`, `age_band`, and `metrics` (e.g. `reason`, `dtw_rows`) so the dashboard can show why. Never a missing file or plain `{}`.

**What's created (per cohort/age_band):**
- **Feature engineering** (in `.../dtw/feature_engineering/`): `dtw_features_{cohort}_{age_band_fname}.csv`, `common_sequences_*.json`
- **Dashboard artifacts** (in `.../dtw/{cohort}/{age_band_fname}/`):
  - `chart_data.json` – routine_comparison, high_risk_trajectories, target_pathway_patterns, times_between_sequences, routine_comparison_counts, event_density_bins (full or empty-state JSON)
  - `sequence_heatmap.json` – code×position counts by drug/icd/cpt (full or empty-state JSON)
  - `plots/trajectory_overview_plot.json` – Plotly overview (full or empty-state JSON)
  - `plots/dtw_trajectory_analysis_{base}.png`, `plots/dtw_sample_trajectories_{base}.png` – PNG fallbacks (when kaleido available)
  - `plots/dtw_trajectory_cluster_1d_{base}.html`, `plots/dtw_trajectory_cluster_3d_{base}.html` – interactive overview

The **manifest** (`dashboard_visual_objects.json`) lists all DTW objects and S3 paths. The dashboard uses **JSON-first** loading: fetches `chart_data.json`, `sequence_heatmap.json`, and `plots/trajectory_overview_plot.json` from static (manifest URLs) first, then PNG/HTML fallbacks from manifest, API only when static fails.

**Runtime:** ~10–30 minutes per cohort/age_band (includes DTW distance matrix computation).

**Research question:** Routine vs no routine appointments → outcomes; sequence-level patterns (drug pathways preceding adverse events). **Date scope:** full pipeline (2016–2019).

In [7]:
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from py_helpers.constants import REQUIRED_COHORTS, DASHBOARD_VISUAL_AGE_BANDS

FORCE_RERUN = True
PARALLEL_WORKERS = 12

# Full dashboard scope (both cohorts, 13-24 … 85-114):
combinations = [
    (c, ab) for c, bands in REQUIRED_COHORTS.items()
    for ab in bands if ab in DASHBOARD_VISUAL_AGE_BANDS
]

print(f"DTW will run {len(combinations)} jobs: {combinations[:3]}...")


DTW will run 14 jobs: [('opioid_ed', '13-24'), ('opioid_ed', '25-44'), ('opioid_ed', '45-54')]...


In [ ]:

try:
    FAIL_FAST
except NameError:
    FAIL_FAST = True  # Stop on first failure; set False to continue (also in config/BupaR cell)
force_flag = ["--force"] if FORCE_RERUN else []

def run_dtw_one(cohort_name, age_band):
    """Run DTW trajectory extraction, alignment, and visualization (three-step process)."""
    # Step 1: Extract trajectories from model_data (filtered by SHAP/FFA)
    r_traj = subprocess.run(
        [str(PYTHON_BIN), str(DTW_TRAJECTORIES_SCRIPT), 
         "--cohort", cohort_name, "--age-band", age_band,
         "--project-root", str(REPO_ROOT)] + force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    if r_traj.returncode != 0:
        return (cohort_name, age_band, "trajectories", r_traj.returncode, r_traj.stdout, r_traj.stderr)
    
    # Step 2: DTW alignment (compute distances to prototypes, identify common sequences)
    r_align = subprocess.run(
        [str(PYTHON_BIN), str(DTW_FEATURES_SCRIPT), 
         "--cohort", cohort_name, "--age-band", age_band,
         "--project-root", str(REPO_ROOT)] + force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    if r_align.returncode != 0:
        return (cohort_name, age_band, "alignment", r_align.returncode, r_align.stdout, r_align.stderr)
    
    # Step 3: Create and publish visualizations
    r_vis = subprocess.run(
        [str(PYTHON_BIN), str(DTW_VISUALS_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band,
         "--project-root", str(REPO_ROOT)] + force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    if r_vis.returncode != 0:
        return (cohort_name, age_band, "visuals", r_vis.returncode, r_vis.stdout, r_vis.stderr)
    
    return (cohort_name, age_band, "success", 0, "", "")

with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as ex:
    futures = {ex.submit(run_dtw_one, c, ab): (c, ab) for c, ab in combinations}
    for fut in as_completed(futures):
        cohort_name, age_band, step, code, stdout, stderr = fut.result()
        if code == 0:
            print(f"  [DTW] {cohort_name} / {age_band} -> SUCCESS")
        else:
            print(f"  [DTW] {cohort_name} / {age_band} -> FAILED at {step} (exit {code})")
            if stderr:
                print(f"    stderr: {stderr[:1500]}{'...' if len(stderr) > 1500 else ''}")
            if FAIL_FAST:
                raise RuntimeError(f"DTW {step} failed: {cohort_name} / {age_band}")
print("DTW done (trajectories + alignment + visuals).")

  [DTW] opioid_ed / 25-44 -> SUCCESS
  [DTW] opioid_ed / 45-54 -> SUCCESS
  [DTW] non_opioid_ed / 65-74 -> SUCCESS
  [DTW] opioid_ed / 55-64 -> SUCCESS
  [DTW] opioid_ed / 13-24 -> SUCCESS
  [DTW] non_opioid_ed / 75-84 -> SUCCESS
  [DTW] opioid_ed / 85-114 -> SUCCESS
  [DTW] non_opioid_ed / 85-114 -> SUCCESS
  [DTW] opioid_ed / 75-84 -> SUCCESS
  [DTW] non_opioid_ed / 55-64 -> SUCCESS
  [DTW] non_opioid_ed / 45-54 -> SUCCESS


In [ ]:
# Convert FI CSVs to JSON for dashboard filters (model, cohort, age_band, all age bands)
from py_helpers.feature_importance_heatmap import build_fi_dashboard_jsons

FI_OUTPUTS_BASE = REPO_ROOT / "3a_feature_importance" / "outputs"
result = build_fi_dashboard_jsons(FI_OUTPUTS_BASE, top_n=50, single_band_top_n=100)
print(f"Wrote {len(result['written'])} JSONs (index + heatmaps + single-age).")
for p in result["written"][:15]:
    print(f"  {p}")
if len(result["written"]) > 15:
    print(f"  ... and {len(result['written']) - 15} more")

### Appointments vs no appointments and extreme-density cohorts

**Research question (N1):** Is there a difference in outcomes for patients without routine appointments vs those with routine care? The DTW tab answers this via **Routine vs No Routine (Outcomes)** and **High-Risk vs Low-Risk Trajectories** (see above). These are shown over the **full pipeline (2016–2019)**, not a single year.

**Extreme-density cohorts** are high-utilizer patients (top ~5% by medical_code transaction density) split out so they do not dominate main models (see `docs/Step4_ModelData/README_model_data_and_extreme_split.md`). For **each cohort and age band**, running extract + DTW (and optionally BupaR) for the extreme-density subgroup lets you compare **routine vs no routine** (outcomes and trajectories) in the high-utilizer subgroup and how **extreme densities** and **extreme-density trajectories** differ across age bands and cohorts. By default the cell below uses the **same (cohort, age_band) combinations** as the main pipeline. Set `EXTREME_COMBINATIONS = []` to skip. Requires **Step 4** (model data) first.

In [ ]:
# Default: same (cohort, age_band) as main pipeline so we get routine vs no routine and
# extreme-density trajectories for every cohort and age band. Set to [] to skip.
# Full DTW pipeline for extreme: extract -> trajectories -> features -> visuals
# (create_dtw_visuals needs dtw_features_* CSV; trajectories uses base cohort allowed_codes when cohort ends with _extreme_density)
EXTREME_COMBINATIONS = combinations  # from config cell above

EXTREME_EXTRACT_SCRIPT = STEP9_ROOT / "dtw" / "extract_extreme_density_cohort.py"
extreme_force_flag = ["--force"] if FORCE_RERUN else []

def run_extreme_one(cohort_name, age_band):
    r0 = subprocess.run(
        [str(PYTHON_BIN), str(EXTREME_EXTRACT_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band],
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    if r0.returncode != 0:
        return (cohort_name, age_band, r0.returncode, None, None, None, r0.stdout, r0.stderr, None, None, None)
    extreme_name = f"{cohort_name}_extreme_density"
    r_traj = subprocess.run(
        [str(PYTHON_BIN), str(DTW_TRAJECTORIES_SCRIPT), "--cohort-name", extreme_name, "--age-band", age_band,
         "--project-root", str(REPO_ROOT)] + extreme_force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    if r_traj.returncode != 0:
        return (cohort_name, age_band, r0.returncode, r_traj.returncode, None, None, r0.stdout, r0.stderr, r_traj.stdout, r_traj.stderr, None)
    r_feat = subprocess.run(
        [str(PYTHON_BIN), str(DTW_FEATURES_SCRIPT), "--cohort", extreme_name, "--age-band", age_band,
         "--project-root", str(REPO_ROOT)] + extreme_force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    if r_feat.returncode != 0:
        return (cohort_name, age_band, r0.returncode, r_traj.returncode, r_feat.returncode, None, r0.stdout, r0.stderr, r_traj.stderr, r_feat.stderr, None)
    r_vis = subprocess.run(
        [str(PYTHON_BIN), str(DTW_VISUALS_SCRIPT), "--cohort-name", extreme_name, "--age-band", age_band,
         "--project-root", str(REPO_ROOT)] + extreme_force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    return (cohort_name, age_band, r0.returncode, r_traj.returncode, r_feat.returncode, r_vis.returncode, r0.stdout, r0.stderr, r_traj.stderr, r_feat.stderr, r_vis.stderr)

if not EXTREME_COMBINATIONS:
    print("EXTREME_COMBINATIONS is empty; skipping extreme-density cohort extraction and DTW.")
else:
    from concurrent.futures import ThreadPoolExecutor, as_completed
    with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as ex:
        futures = {ex.submit(run_extreme_one, c, ab): (c, ab) for c, ab in EXTREME_COMBINATIONS}
        for fut in as_completed(futures):
            result = fut.result()
            cohort_name, age_band = result[0], result[1]
            c0, c_traj, c_feat, c_vis = result[2], result[3], result[4], result[5]
            print(f"  [Extreme] {cohort_name} / {age_band} -> extract={c0}, traj={c_traj}, feat={c_feat}, vis={c_vis}")
            if c0 != 0:
                print(f"    extract_extreme_density_cohort failed (exit {c0})")
                if len(result) > 7 and result[7]:
                    print("    stderr:", (result[7][:1500] + "..." if len(result[7]) > 1500 else result[7]))
                if FAIL_FAST:
                    raise RuntimeError(f"Extract extreme cohort failed: {cohort_name} / {age_band}")
            if c_traj is not None and c_traj != 0:
                print(f"    create_dtw_trajectories failed (exit {c_traj})")
                if FAIL_FAST:
                    raise RuntimeError(f"DTW create_dtw_trajectories failed: {cohort_name}_extreme_density / {age_band}")
            if c_feat is not None and c_feat != 0:
                print(f"    create_dtw_features failed (exit {c_feat})")
                if FAIL_FAST:
                    raise RuntimeError(f"DTW create_dtw_features failed: {cohort_name}_extreme_density / {age_band}")
            if c_vis is not None and c_vis != 0:
                print(f"    create_dtw_visuals failed (exit {c_vis})")
                if len(result) > 10 and result[10]:
                    print("    stderr:", (result[10][:800] + "..." if len(result[10]) > 800 else result[10]))
                if FAIL_FAST:
                    raise RuntimeError(f"DTW create_dtw_visuals failed: {cohort_name}_extreme_density / {age_band}")
    print(f"Done: extreme-density extract + DTW (trajectories + features + visuals) for {len(EXTREME_COMBINATIONS)} combinations (parallel).")

## Run FP-Growth (itemsets, Plotly network HTML, S3 upload)

FP-Growth uses **SHAP/FFA-refined** model data: inputs come from `4_model_data` (built from Step 3b `cohort_feature_importance.csv`). **Item types included: drugs (`drug_name`), ICD diagnosis codes (`icd_code`), and CPT procedure codes (`cpt_code`)** (plus combined `medical_code`). For each cohort/age band this step: (1) ensures itemsets exist, (2) creates PNGs and **Plotly interactive network HTML**, (3) **saves locally** to `10_risk_dashboard/visualizations/fpgrowth/{cohort}/{age_band_fname}/plots/` and `.../data/` (notebook 5 Step 6 syncs to S3 `visualizations/fpgrowth/{cohort}/{age_band}/`). The dashboard then shows the **network plot for the user-selected cohort** via the `/visualizations/fpgrowth` API. **FP-Growth itemsets and rules are not used for feature engineering** (same as DTW and BupaR); they are computed for dashboard visualization and analysis.

Run the cell below in parallel (FPGROWTH_WORKERS at a time; builds itemsets and Plotly HTML, saves to final local destination). **Exit 0** = itemsets/plots produced for that cohort/age_band; **exit 1** = no outputs (e.g. model_data missing or too few transactions). Logs: `logs/9_fpgrowth/` (EC2) and S3 `s3://pgx-repository/9_fpgrowth_log/{cohort}/{age_band}/` (mirrored on `log_summary()`).

In [ ]:
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

try:
    FAIL_FAST
except NameError:
    FAIL_FAST = True
try:
    FPGROWTH_WORKERS
except NameError:
    # Maximize CPU utilization: use all cores up to number of combinations
    import os
    FPGROWTH_WORKERS = min(os.cpu_count() or 4, len(combinations))
force_flag = ["--force"] if FORCE_RERUN else []

def run_fpgrowth_one(cohort_name, age_band):
    r = subprocess.run(
        [str(PYTHON_BIN), str(FPGROWTH_VISUALS_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band] + force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    return (cohort_name, age_band, r.returncode, r.stdout, r.stderr)

with ThreadPoolExecutor(max_workers=FPGROWTH_WORKERS) as ex:
    futures = {ex.submit(run_fpgrowth_one, c, ab): (c, ab) for c, ab in combinations}
    for fut in as_completed(futures):
        cohort_name, age_band, code, stdout, stderr = fut.result()
        print(f"  [FP-Growth] {cohort_name} / {age_band} -> exit {code}")
        if code != 0:
            ab_f = age_band.replace("-", "_")
            print(f"    No itemsets produced (exit 1). Check logs/9_fpgrowth/create_fpgrowth_visuals_{cohort_name}_{ab_f}_*.log or s3://pgx-repository/9_fpgrowth_log/{cohort_name}/{age_band}/")
            if stderr:
                print("    stderr:", (stderr[:1500] + "..." if len(stderr) > 1500 else stderr))
            if stdout:
                print("    stdout:", (stdout[:800] + "..." if len(stdout) > 800 else stdout))
            if FAIL_FAST:
                raise RuntimeError(f"FP-Growth failed: {cohort_name} / {age_band}")
print("FP-Growth done.")

## View visualization outputs

**Outputs are not in the repo** — they are created when you run the FI heatmaps, BupaR, DTW, FP-Growth, Causal, and Cohort PGx cells above. Paths: **`10_risk_dashboard/visualizations/`** with **`bupar/`**, **`dtw/`**, **`fpgrowth/`**, **`feature_importance/`**, **`causal/`**, **`cohort_pgx/`** (each has cohort/age_band or cohort-only structure; plots in `plots/` or at root). Run the cell below **after** those steps to list paths and preview sample **JSON** (truncated), **PNG** (image), and **HTML** (link). If you see "No outputs yet", run the pipeline cells above first.

In [ ]:
# Where outputs live and sample preview (run after FI heatmaps, BupaR, DTW, FP-Growth, Scenario, Cohort PGx)
import html
import json
from pathlib import Path
from IPython.display import display, Image, HTML, IFrame

def _first_plots_dir(base: Path, subdir: str, use_outputs: bool = False):
    """Find first cohort/age_band/plots dir. Try direct (bupar/dtw) then outputs/ (fpgrowth)."""
    roots = [base / subdir / "outputs", base / subdir] if use_outputs else [base / subdir, base / subdir / "outputs"]
    for root in roots:
        if not root.exists():
            continue
        for cohort in sorted(root.iterdir()):
            if not cohort.is_dir() or cohort.name.startswith("."):
                continue
            for age in sorted(cohort.iterdir()):
                if not age.is_dir():
                    continue
                plots = age / "plots"
                if plots.exists():
                    return plots
    return None

def _preview_json(path: Path, max_lines: int = 25):
    """Show truncated JSON (first max_lines of pretty-printed), HTML-escaped for safe display."""
    try:
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        raw = json.dumps(data, indent=2)
        lines = raw.split("\n")[:max_lines]
        text = "\n".join(lines) + ("\n..." if len(raw.split("\n")) > max_lines else "")
        return html.escape(text)
    except Exception:
        return "(could not parse JSON)"

base = REPO_ROOT / "10_risk_dashboard" / "visualizations"
print("Output root:", base)
print()

# --- BupaR, DTW, FP-Growth (plots: PNG, HTML, JSON) ---
for name, subdir in [("BupaR", "bupar"), ("DTW", "dtw"), ("FP-Growth", "fpgrowth")]:
    plots_dir = _first_plots_dir(base, subdir, use_outputs=(subdir == "fpgrowth"))
    print(f"--- {name} ---")
    if not plots_dir:
        print(f"  No outputs yet. Run the \"Run {name}\" cell above, then re-run this cell.")
        print(f"  Paths: {base / subdir}/<cohort>/<age_band>/plots/ or .../outputs/...")
        continue
    print(f"  Sample dir: {plots_dir}")
    pngs = sorted(plots_dir.glob("*.png"))
    htmls = sorted(plots_dir.glob("*.html"))
    jsons = sorted(plots_dir.glob("*.json"))
    print(f"  PNGs: {len(pngs)}, HTMLs: {len(htmls)}, JSONs: {len(jsons)}")
    if pngs:
        display(HTML(f"<b>{name} (sample PNG)</b>"))
        display(Image(filename=str(pngs[0]), width=600))
    if jsons:
        display(HTML(f"<b>{name} (sample JSON preview)</b>"))
        display(HTML(f"<pre>{_preview_json(jsons[0])}</pre>"))
    if htmls:
        display(HTML(f"<b>{name} (sample HTML)</b>"))
        display(HTML(f'<a href="file:///{htmls[0].resolve().as_posix()}" target="_blank">Open in browser</a>'))
    print()

# --- Feature importance (PNG + JSON under feature_importance/ and feature_importance/{cohort}/, combined/) ---
fi_base = base / "feature_importance"
print("--- Feature importance ---")
if fi_base.exists():
    fi_pngs = list(fi_base.glob("*.png")) + list(fi_base.rglob("*.png"))
    fi_jsons = list(fi_base.glob("*.json")) + list(fi_base.rglob("*.json"))
    print(f"  Path: {fi_base}")
    print(f"  PNGs: {len(fi_pngs)}, JSONs: {len(fi_jsons)}")
    if fi_pngs:
        display(HTML("<b>Feature importance (sample PNG)</b>"))
        display(Image(filename=str(fi_pngs[0]), width=600))
    if fi_jsons:
        display(HTML("<b>Feature importance (sample JSON preview)</b>"))
        display(HTML(f"<pre>{_preview_json(fi_jsons[0])}</pre>"))
else:
    print("  No outputs yet. Run the FI heatmaps cell and copy step above.")
print()

# --- Causal (dashboard_data.json per cohort/age_band) ---
causal_base = base / "causal"
print("--- Causal ---")
causal_jsons = list(causal_base.rglob("dashboard_data.json")) if causal_base.exists() else []
if causal_jsons:
    print(f"  Path: {causal_base}/<cohort>/<age_band_fname>/")
    print(f"  JSONs: {len(causal_jsons)}")
    display(HTML("<b>Causal (sample JSON preview)</b>"))
    display(HTML(f"<pre>{_preview_json(causal_jsons[0])}</pre>"))
else:
    print("  No outputs yet. Run the combine SHAP+FFA / Causal step above.")
print()

# --- Cohort PGx (network HTML + JSON under cohort_pgx/networks/) ---
pgx_base = base / "cohort_pgx" / "networks"
print("--- Cohort PGx ---")
if pgx_base.exists():
    pgx_htmls = list(pgx_base.rglob("*.html"))
    pgx_jsons = list(pgx_base.rglob("*.json"))
    print(f"  Path: {pgx_base}/<cohort>/<age_band_fname>/")
    print(f"  HTMLs: {len(pgx_htmls)}, JSONs: {len(pgx_jsons)}")
    if pgx_htmls:
        display(HTML("<b>Cohort PGx (sample HTML)</b>"))
        display(HTML(f'<a href="file:///{pgx_htmls[0].resolve().as_posix()}" target="_blank">Open in browser</a>'))
    if pgx_jsons:
        display(HTML("<b>Cohort PGx (sample JSON preview)</b>"))
        display(HTML(f"<pre>{_preview_json(pgx_jsons[0])}</pre>"))
else:
    print("  No outputs yet. Run the Cohort PGx step above.")
print()
print("All outputs: 10_risk_dashboard/visualizations/{bupar,dtw,fpgrowth,feature_importance,causal,cohort_pgx/networks}/...")

## PGx Patient Card Setup (Optional)

The dashboard includes a **PGx Patient Card** feature (Tab 2) that generates personalized pharmacogenomic cards from genetic variants. The Lambda function (`POST /pgx/card`) uses **CPIC gene-drug pairs** data to match variants to drugs requiring dosing modifications.

**Optional enhancement**: Add **PharmGKB VIP URLs** and **QR codes** to patient cards for richer gene information. Run the cells below to:
1. Fetch PharmGKB VIP gene data (uses current API)
2. Generate QR codes pointing to ClinPGx VIP pages
3. Build unified PGx database for Lambda integration

**Note**: The Lambda already works with CPIC data alone. This step adds PharmGKB integration for enhanced cards.

**Python migration (2026)**: Old R scripts (`PGx.Rmd`, `Build_PGx_Database.Rmd`) are deprecated due to PharmGKB API changes. See `pgx-patient-card/README_PYTHON.md` for details.

In [ ]:
# PGx Patient Card Setup - Paths
PGX_CARD_DIR = REPO_ROOT / "pgx-patient-card"
PGX_DATA_DIR = PGX_CARD_DIR / "data"
PGX_QR_DIR = PGX_CARD_DIR / "qr_codes"

# Scripts
DOWNLOAD_CPIC_SCRIPT = PGX_CARD_DIR / "download_cpic_excel.py"
FETCH_PHARMGKB_SCRIPT = PGX_CARD_DIR / "fetch_pharmgkb_data.py"
GENERATE_QR_SCRIPT = PGX_CARD_DIR / "generate_pgx_qr_codes.py"
BUILD_DATABASE_SCRIPT = PGX_CARD_DIR / "build_pgx_database.py"

# Data files
CPIC_EXCEL = PGX_DATA_DIR / "cpic_gene-drug_pairs.xlsx"
VIP_JSON = PGX_DATA_DIR / "pharmgkb_vip_genes.json"
QR_MAPPINGS_JSON = PGX_DATA_DIR / "qr_code_mappings.json"
PGX_DATABASE_DIR = PGX_DATA_DIR / "pgx_database"

print(f"PGx card directory: {PGX_CARD_DIR}")
print(f"Data directory: {PGX_DATA_DIR}")
print(f"QR codes directory: {PGX_QR_DIR}")
print()
print("Files:")
print(f"  CPIC Excel: {CPIC_EXCEL} {'✓' if CPIC_EXCEL.exists() else '✗'}")
print(f"  VIP JSON: {VIP_JSON} {'✓' if VIP_JSON.exists() else '✗'}")
print(f"  QR mappings: {QR_MAPPINGS_JSON} {'✓' if QR_MAPPINGS_JSON.exists() else '✗'}")
print(f"  Database: {PGX_DATABASE_DIR} {'✓' if PGX_DATABASE_DIR.exists() else '✗'}")

### Step 1: Download CPIC Data

Download the latest CPIC gene-drug pairs Excel file. This is the **primary data source** for the PGx card feature.

**URL**: `https://files.cpicpgx.org/data/report/current/pair/cpic_gene-drug_pairs.xlsx`

**Verified**: February 2026 (31KB, last modified Feb 5, 2026)

In [ ]:
# Download CPIC gene-drug pairs Excel file (uses download_cpic_excel.py with SSL fallback)
if CPIC_EXCEL.exists() and CPIC_EXCEL.stat().st_size > 0:
    print(f"✓ CPIC Excel already exists: {CPIC_EXCEL}")
    print(f"  Size: {CPIC_EXCEL.stat().st_size / 1024:.1f} KB")
else:
    result = subprocess.run(
        [str(PYTHON_BIN), str(DOWNLOAD_CPIC_SCRIPT)],
        cwd=str(PGX_CARD_DIR),
        capture_output=True,
        text=True,
    )
    if result.returncode == 0:
        if result.stdout:
            print(result.stdout)
    else:
        print(result.stdout or "")
        print(result.stderr or "")
        raise RuntimeError(f"CPIC download failed (exit {result.returncode})")

### Step 2: Fetch PharmGKB VIP Data

Fetch VIP (Very Important Pharmacogene) data from PharmGKB API. This adds **ClinPGx VIP URLs** for genes.

**API**: `https://api.pharmgkb.org/v1/data/gene?symbol=...` (documented in Postman)

**Output**: `data/pharmgkb_vip_genes.json` with VIP URLs for 20 genes

In [ ]:
# Fetch PharmGKB VIP gene data
if VIP_JSON.exists():
    print(f"✓ VIP data already exists: {VIP_JSON}")
    import json
    with open(VIP_JSON) as f:
        vip_data = json.load(f)
    print(f"  Contains {len(vip_data)} VIP genes")
else:
    print("Fetching PharmGKB VIP gene data...")
    print(f"  Running: {FETCH_PHARMGKB_SCRIPT}")
    result = subprocess.run(
        [str(PYTHON_BIN), str(FETCH_PHARMGKB_SCRIPT)],
        cwd=str(PGX_CARD_DIR),
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print("✓ VIP data fetched successfully")
        print(result.stdout)
    else:
        print(f"✗ Failed (exit {result.returncode})")
        print(result.stderr)

### Step 3: Generate QR Codes

Generate QR codes for ClinPGx VIP pages. Each gene gets a QR code pointing to `https://www.clinpgx.org/vip/{PA_ID}/overview`.

**Prerequisites**: `pip install qrcode[pil]`

**Output**: `qr_codes/{GENE}.png` (20 QR code images, 200x200 px)

In [ ]:
# Generate QR codes for VIP pages
if QR_MAPPINGS_JSON.exists() and PGX_QR_DIR.exists() and len(list(PGX_QR_DIR.glob("*.png"))) > 0:
    print(f"✓ QR codes already exist: {PGX_QR_DIR}")
    print(f"  Contains {len(list(PGX_QR_DIR.glob('*.png')))} QR code images")
else:
    print("Generating QR codes for VIP pages...")
    print(f"  Running: {GENERATE_QR_SCRIPT}")
    result = subprocess.run(
        [str(PYTHON_BIN), str(GENERATE_QR_SCRIPT)],
        cwd=str(PGX_CARD_DIR),
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print("✓ QR codes generated successfully")
        print(result.stdout)
    else:
        print(f"✗ Failed (exit {result.returncode})")
        print(result.stderr)
        if "ModuleNotFoundError" in result.stderr and "qrcode" in result.stderr:
            print("\nInstall missing dependency:")
            print("  pip install qrcode[pil]")

### Step 4: Build Unified PGx Database

Merge CPIC, PharmGKB VIP, and QR code data into a unified database for patient card generation.

**Output**:
- `data/pgx_database/pgx_database.csv` - Merged data (CSV)
- `data/pgx_database/pgx_database.json` - Merged data (JSON)
- `data/pgx_database/pgx_database.xlsx` - Merged data (Excel)
- `data/pgx_database/pgx_database_summary.json` - Summary statistics

In [ ]:
# Build unified PGx database
if PGX_DATABASE_DIR.exists() and (PGX_DATABASE_DIR / "pgx_database.csv").exists():
    print(f"✓ PGx database already exists: {PGX_DATABASE_DIR}")
    import json
    summary_path = PGX_DATABASE_DIR / "pgx_database_summary.json"
    if summary_path.exists():
        with open(summary_path) as f:
            summary = json.load(f)
        print("\nDatabase summary:")
        for key, value in summary.items():
            print(f"  {key.replace('_', ' ').title()}: {value}")
else:
    print("Building unified PGx database...")
    print(f"  Running: {BUILD_DATABASE_SCRIPT}")
    result = subprocess.run(
        [str(PYTHON_BIN), str(BUILD_DATABASE_SCRIPT)],
        cwd=str(PGX_CARD_DIR),
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print("✓ PGx database built successfully")
        print(result.stdout)
    else:
        print(f"✗ Failed (exit {result.returncode})")
        print(result.stderr)

### Integration with Lambda

To integrate VIP URLs with the Lambda function (`POST /pgx/card`):

1. **Copy VIP data to Lambda container** (in Dockerfile or build script):
   ```dockerfile
   COPY pgx-patient-card/data/pharmgkb_vip_genes.json ${LAMBDA_TASK_ROOT}/data/
   ```

2. **Load at Lambda startup** (in `lambda_function.py`):
   ```python
   VIP_URL_CACHE = {}
   
   def load_vip_urls():
       vip_path = '/var/task/data/pharmgkb_vip_genes.json'
       if os.path.exists(vip_path):
           with open(vip_path) as f:
               vip_data = json.load(f)
           VIP_URL_CACHE = {item['gene'].upper(): item['vip_url'] for item in vip_data}
   
   load_vip_urls()  # Call at module level
   ```

3. **Add VIP URLs to card response** (in `generate_pgx_card()`):
   ```python
   for gene_entry in genes_processed:
       if gene_entry['gene'] in VIP_URL_CACHE:
           gene_entry['vip_url'] = VIP_URL_CACHE[gene_entry['gene']]
   ```

See `pgx-patient-card/README_PYTHON.md` for full Lambda integration details.

## Cohort PGx Network Topology

The dashboard includes a **Cohort PGx** tab that combines PharmGKB VIP reports for all important genes in a cohort with network topology analysis. This feature:

1. **Extracts PGx genes** from SHAP/FFA feature importance (top N genes from Step 3b or notebook 3)
2. **Fetches PharmGKB VIP reports** with clinical annotations and drug interactions
3. **Builds network topology** using:
   - **pytextrank**: Key phrase extraction and entity recognition from VIP text
   - **AWS Comprehend** (optional): Medical entity recognition and key phrase extraction
   - **Network analysis**: Gene-drug-phenotype relationships visualized as Plotly interactive graph

**Output**: Interactive network visualization showing how cohort-specific genes relate to drugs and phenotypes, plus exportable node/edge data for further analysis.

**Research value**: Identifies gene-drug interaction patterns specific to each cohort/age band, revealing pharmacogenomic mechanisms underlying adverse event risk.

In [ ]:
# Cohort PGx Network Topology - Paths
COHORT_PGX_DIR = STEP9_ROOT / "cohort_pgx"
COHORT_PGX_REPORTS_DIR = VISUAL_ROOT / "cohort_pgx" / "reports"
COHORT_PGX_NETWORKS_DIR = VISUAL_ROOT / "cohort_pgx" / "networks"

# Scripts
FETCH_VIP_REPORTS_SCRIPT = COHORT_PGX_DIR / "fetch_vip_reports.py"
FETCH_PUBMED_CITATIONS_SCRIPT = COHORT_PGX_DIR / "fetch_pubmed_citations.py"
BUILD_NETWORK_TOPOLOGY_SCRIPT = COHORT_PGX_DIR / "build_network_topology.py"

print(f"Cohort PGx directory: {COHORT_PGX_DIR}")
print(f"Scripts:")
print(f"  Fetch VIP reports: {FETCH_VIP_REPORTS_SCRIPT} {'✓' if FETCH_VIP_REPORTS_SCRIPT.exists() else '✗'}")
print(f"  Fetch PubMed cites: {FETCH_PUBMED_CITATIONS_SCRIPT} {'✓' if FETCH_PUBMED_CITATIONS_SCRIPT.exists() else '✗'}")
print(f"  Build network: {BUILD_NETWORK_TOPOLOGY_SCRIPT} {'✓' if BUILD_NETWORK_TOPOLOGY_SCRIPT.exists() else '✗'}")
print()
print(f"Outputs:")
print(f"  Reports: {COHORT_PGX_REPORTS_DIR}")
print(f"  Networks: {COHORT_PGX_NETWORKS_DIR}")

# Config
COHORT_PGX_TOP_N = 50  # Top N genes from feature importance
COHORT_PGX_COHORTS = combinations  # Use same cohort/age_band combinations as main pipeline
USE_COMPREHEND = True  # Set False to skip AWS Comprehend (pytextrank only)

print()
print(f"Config:")
print(f"  Top N genes per cohort: {COHORT_PGX_TOP_N}")
print(f"  Cohort combinations: {len(COHORT_PGX_COHORTS)}")
print(f"  AWS Comprehend: {'Enabled' if USE_COMPREHEND else 'Disabled (pytextrank only)'}")
# Utilization-density bins for per-bin PGx tasks (same as BupaR/scenario cells)
try:
    _SCENARIO_DENSITY_BINS
except NameError:
    from py_helpers.event_density_utils import DENSITY_BINS as _SCENARIO_DENSITY_BINS



### Step 1: Fetch PharmGKB VIP Reports

For each cohort/age band, extract top N genes from feature importance and fetch VIP reports from PharmGKB API.

**Prerequisites**:
- Feature importance data (Step 3b or notebook 3 combined_importance.csv)
- `pip install beautifulsoup4` for VIP page text extraction

**Output**: `{cohort}_{age_band}_vip_reports.json` with gene metadata, clinical annotations, and VIP page text

**Runtime**: ~1-2 minutes per cohort/age band (API rate limited to 0.5s between requests)

In [ ]:
# Fetch VIP reports for all cohort/age_band combinations
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

def run_fetch_vip_reports(cohort_name, age_band, bin_name=None):
    """Fetch PharmGKB VIP reports for one cohort/age band (optionally per density bin)."""
    args = [
        str(PYTHON_BIN), str(FETCH_VIP_REPORTS_SCRIPT),
        "--cohort", cohort_name,
        "--age-band", age_band,
        "--top-n", str(COHORT_PGX_TOP_N),
        "--project-root", str(REPO_ROOT),
        "--output-dir", str(COHORT_PGX_REPORTS_DIR)
    ]
    if bin_name:
        args += ["--bin", bin_name]
    # Skip VIP page fetching if needed (faster but less text data)
    # args.append("--no-vip-pages")
    result = subprocess.run(args, cwd=str(REPO_ROOT), capture_output=True, text=True)
    return (cohort_name, age_band, bin_name, result.returncode, result.stdout, result.stderr)

# Build task list: full-cohort + per-bin for each combination
_pgx_tasks = [
    (c, ab, None) for c, ab in COHORT_PGX_COHORTS
] + [
    (c, ab, b) for c, ab in COHORT_PGX_COHORTS for b in _SCENARIO_DENSITY_BINS
]
print(f"Fetching VIP reports for {len(COHORT_PGX_COHORTS)} cohort/age_band combinations + {len(_SCENARIO_DENSITY_BINS)} bins each...")
print(f"  Top {COHORT_PGX_TOP_N} genes per cohort; total tasks: {len(_pgx_tasks)}")
print()

# Run in parallel (max 2 at a time to respect API rate limits)
MAX_WORKERS_PGX = 2  # Conservative to avoid API throttling

with ThreadPoolExecutor(max_workers=MAX_WORKERS_PGX) as ex:
    futures = {ex.submit(run_fetch_vip_reports, c, ab, b): (c, ab, b) for c, ab, b in _pgx_tasks}
    for fut in as_completed(futures):
        cohort_name, age_band, bin_name, code, stdout, stderr = fut.result()
        label = f"{cohort_name} / {age_band}" + (f" [{bin_name}]" if bin_name else "")
        if code == 0:
            print(f"  [VIP Reports] {label} -> SUCCESS")
        else:
            print(f"  [VIP Reports] {label} -> FAILED (exit {code})")
            if stderr:
                print(f"    stderr: {stderr[:800]}{'...' if len(stderr) > 800 else ''}")
            if FAIL_FAST:
                raise RuntimeError(f"Fetch VIP reports failed: {label}")

print("\n✓ VIP reports fetched for all cohorts (full-cohort + per-bin)")
print(f"  Output: {COHORT_PGX_REPORTS_DIR}")

### Step 1.5: Fetch PubMed Citations (Literature QA)

For each cohort/age band (and per density bin), query NCBI PubMed E-utilities to retrieve
supporting literature for the PGx genes found in the VIP reports.  Follows the same
search methodology as `lit_review/lit_review.qmd` (`search_pubmed_all` pattern):
  - Date range: last 5 years  (`{year-5}:{year}[PDAT]`)
  - Query 1: gene + pharmacogenomics MeSH  (general clinical evidence)
  - Query 2: gene + cohort-context keyword  (opioid / emergency department)
  - XML efetch to capture PMC IDs + BioC JSON full-text URL

**Prerequisites**: VIP reports from Step 1 must exist.

**Output per cohort/age band**: `pubmed_citations.json` in the networks output directory
(same location as `network_topology.html`), synced to S3 by `sync_cohort_pgx_to_s3.py`.

**Rate limiting**: 3 req/s without API key; set env var `NCBI_API_KEY` to raise to 10 req/s.
**Runtime**: ~1–3 min per cohort/age band (NCBI polite limit).

In [ ]:
# Fetch PubMed citations for all cohort/age_band combinations (full-cohort + per-bin)
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

# Ensure Cohort PGx paths are defined (run "Cohort PGx Network Topology - Paths" cell first)
try:
    FETCH_PUBMED_CITATIONS_SCRIPT
except NameError:
    from pathlib import Path
    _r = globals().get("REPO_ROOT", Path.cwd())
    _step9 = _r / "10_risk_dashboard"
    _visual = _step9 / "visualizations"
    COHORT_PGX_DIR = _step9 / "cohort_pgx"
    COHORT_PGX_REPORTS_DIR = _visual / "cohort_pgx" / "reports"
    COHORT_PGX_NETWORKS_DIR = _visual / "cohort_pgx" / "networks"
    FETCH_PUBMED_CITATIONS_SCRIPT = COHORT_PGX_DIR / "fetch_pubmed_citations.py"

try:
    FAIL_FAST
except NameError:
    FAIL_FAST = True  # default; set in Config/BupaR cell when running full pipeline

# Optional NCBI API key (set env var NCBI_API_KEY to use; raises rate limit to 10 req/s)
import os as _os
_NCBI_API_KEY = _os.environ.get("NCBI_API_KEY") or None

def run_fetch_pubmed_citations(cohort_name, age_band, bin_name=None):
    """Fetch PubMed citations + radar data for one cohort/age band (optionally per density bin)."""
    age_band_fname = age_band.replace("-", "_")
    bin_suffix = f"_{bin_name}" if bin_name else ""
    reports_file = COHORT_PGX_REPORTS_DIR / f"{cohort_name}_{age_band_fname}{bin_suffix}_vip_reports.json"
    output_dir = (
        COHORT_PGX_NETWORKS_DIR / cohort_name / age_band_fname / "density" / bin_name
        if bin_name
        else COHORT_PGX_NETWORKS_DIR / cohort_name / age_band_fname
    )

    if not reports_file.exists():
        return (cohort_name, age_band, bin_name, -1, f"Reports file not found: {reports_file}", "")

    args = [
        str(PYTHON_BIN), str(FETCH_PUBMED_CITATIONS_SCRIPT),
        "--cohort", cohort_name,
        "--age-band", age_band,
        "--reports", str(reports_file),
        "--output-dir", str(output_dir),
        "--project-root", str(REPO_ROOT),
    ]
    if bin_name:
        args += ["--bin", bin_name]
    if _NCBI_API_KEY:
        args += ["--ncbi-api-key", _NCBI_API_KEY]

    result = subprocess.run(args, cwd=str(REPO_ROOT), capture_output=True, text=True)
    return (cohort_name, age_band, bin_name, result.returncode, result.stdout, result.stderr)

# Task list mirrors VIP reports: full-cohort + per-bin for each combination
_pubmed_tasks = [
    (c, ab, None) for c, ab in COHORT_PGX_COHORTS
] + [
    (c, ab, b) for c, ab in COHORT_PGX_COHORTS for b in _SCENARIO_DENSITY_BINS
]
print(f"Fetching PubMed citations for {len(COHORT_PGX_COHORTS)} cohort/age_band combinations + {len(_SCENARIO_DENSITY_BINS)} bins each...")
print(f"  NCBI API key: {'set (10 req/s)' if _NCBI_API_KEY else 'not set (3 req/s polite limit)'}; total tasks: {len(_pubmed_tasks)}")
print()

# Max 2 concurrent tasks to respect NCBI rate limits (same as VIP reports step)
MAX_WORKERS_PUBMED = 2

with ThreadPoolExecutor(max_workers=MAX_WORKERS_PUBMED) as ex:
    futures = {ex.submit(run_fetch_pubmed_citations, c, ab, b): (c, ab, b) for c, ab, b in _pubmed_tasks}
    for fut in as_completed(futures):
        cohort_name, age_band, bin_name, code, stdout, stderr = fut.result()
        label = f"{cohort_name} / {age_band}" + (f" [{bin_name}]" if bin_name else "")
        if code == 0:
            print(f"  [PubMed Citations] {label} -> SUCCESS")
        else:
            print(f"  [PubMed Citations] {label} -> FAILED (exit {code})")
            if stderr:
                print(f"    stderr: {stderr[:800]}{'...' if len(stderr) > 800 else ''}")
            if FAIL_FAST:
                raise RuntimeError(f"Fetch PubMed citations failed: {label}")

print("\n✓ PubMed citations + radar data fetched for all cohorts (full-cohort + per-bin)")
print(f"  Output: {COHORT_PGX_NETWORKS_DIR} (pubmed_citations.json + pgx_radar_data.json alongside network_topology.html)")

### Step 2: Build Network Topology

Build interactive network topology from VIP reports using pytextrank and AWS Comprehend.

**Prerequisites**:
- `pip install spacy pytextrank networkx plotly`
- `python -m spacy download en_core_web_sm` (spaCy model)
- `pip install boto3` (optional, for AWS Comprehend)

**Output per cohort/age band**:
- `network_topology.html` - Interactive Plotly network visualization
- `network_nodes.csv` - Node data (genes, drugs, phenotypes)
- `network_edges.csv` - Edge data (relationships and weights)
- `key_phrases.json` - Extracted key phrases per gene
- `network_stats.json` - Network statistics (density, degree, etc.)

**Runtime**: ~2-5 minutes per cohort/age band (depends on text volume and Comprehend usage)

In [ ]:
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

def run_build_network(cohort_name, age_band, bin_name=None):
    """Build network topology for one cohort/age band (optionally per density bin)."""
    age_band_fname = age_band.replace("-", "_")
    bin_suffix = f"_{bin_name}" if bin_name else ""
    reports_file = COHORT_PGX_REPORTS_DIR / f"{cohort_name}_{age_band_fname}{bin_suffix}_vip_reports.json"
    output_dir = (
        COHORT_PGX_NETWORKS_DIR / cohort_name / age_band_fname / "density" / bin_name
        if bin_name
        else COHORT_PGX_NETWORKS_DIR / cohort_name / age_band_fname
    )

    if not reports_file.exists():
        return (cohort_name, age_band, bin_name, -1, f"Reports file not found: {reports_file}", "")

    args = [
        str(PYTHON_BIN), str(BUILD_NETWORK_TOPOLOGY_SCRIPT),
        "--reports", str(reports_file),
        "--output-dir", str(output_dir),
        "--cohort", cohort_name,
        "--age-band", age_band,
    ]
    if bin_name:
        args += ["--bin", bin_name]

    if not USE_COMPREHEND:
        args.append("--no-comprehend")

    result = subprocess.run(args, cwd=str(REPO_ROOT), capture_output=True, text=True)
    return (cohort_name, age_band, bin_name, result.returncode, result.stdout, result.stderr)

# Build task list: full-cohort + per-bin for each combination
_network_tasks = [
    (c, ab, None) for c, ab in COHORT_PGX_COHORTS
] + [
    (c, ab, b) for c, ab in COHORT_PGX_COHORTS for b in _SCENARIO_DENSITY_BINS
]
print(f"Building network topology for {len(COHORT_PGX_COHORTS)} cohort/age_band combinations + {len(_SCENARIO_DENSITY_BINS)} bins each...")
print(f"  AWS Comprehend: {'Enabled' if USE_COMPREHEND else 'Disabled'}; total tasks: {len(_network_tasks)}")
print()

# Run in parallel (max 4 at a time)
MAX_WORKERS_NETWORK = 4

with ThreadPoolExecutor(max_workers=MAX_WORKERS_NETWORK) as ex:
    futures = {ex.submit(run_build_network, c, ab, b): (c, ab, b) for c, ab, b in _network_tasks}
    for fut in as_completed(futures):
        cohort_name, age_band, bin_name, code, stdout, stderr = fut.result()
        label = f"{cohort_name} / {age_band}" + (f" [{bin_name}]" if bin_name else "")
        if code == 0:
            print(f"  [Network Topology] {label} -> SUCCESS")
        else:
            print(f"  [Network Topology] {label} -> FAILED (exit {code})")
            if stderr:
                print(f"    stderr: {stderr[:800]}{'...' if len(stderr) > 800 else ''}")
            if stdout and "not found" in stdout.lower():
                print(f"    {stdout[:400]}{'...' if len(stdout) > 400 else ''}")
            if FAIL_FAST:
                raise RuntimeError(f"Build network topology failed: {label}")

print("\n✓ Network topology built for all cohorts (full-cohort + per-bin)")
print(f"  Output: {COHORT_PGX_NETWORKS_DIR}")

### View Cohort PGx Network Outputs

View network topology visualizations and statistics for all cohorts.

In [ ]:
# View Cohort PGx network outputs
from pathlib import Path
from IPython.display import display, HTML, IFrame
import json

print(f"Cohort PGx Network Outputs: {COHORT_PGX_NETWORKS_DIR}")
print()

if not COHORT_PGX_NETWORKS_DIR.exists():
    print("✗ No outputs yet. Run the cells above to fetch VIP reports and build networks.")
else:
    # Find all network topology HTML files
    html_files = sorted(COHORT_PGX_NETWORKS_DIR.glob("*/*/network_topology.html"))
    
    if not html_files:
        print("✗ No network topology files found. Check if build step completed successfully.")
    else:
        print(f"Found {len(html_files)} network topology visualizations\n")
        
        # Show first one as example
        first_html = html_files[0]
        cohort_name = first_html.parent.parent.name
        age_band = first_html.parent.name.replace("_", "-")
        
        print(f"Example: {cohort_name} / {age_band}")
        print(f"  HTML: {first_html}")
        
        # Show statistics
        stats_file = first_html.parent / "network_stats.json"
        if stats_file.exists():
            with open(stats_file) as f:
                stats = json.load(f)
            print(f"\nNetwork Statistics:")
            for key, value in stats.items():
                print(f"  {key.replace('_', ' ').title()}: {value}")
        
        print(f"\nInteractive visualization:")
        display(HTML(f'<a href="file:///{first_html.resolve().as_posix()}" target="_blank">Open {cohort_name}/{age_band} network in browser</a>'))
        
        # List all outputs
        print(f"\nAll network outputs:")
        for html_file in html_files:
            cohort = html_file.parent.parent.name
            age = html_file.parent.name.replace("_", "-")
            print(f"  - {cohort} / {age}: {html_file.parent}")

print(f"\nDashboard integration: Build step uploads to S3 automatically (same as BupaR/DTW/FP-Growth).")
print(f"  S3 path: {{S3_DASHBOARD_PREFIX}}/cohort_pgx/networks/{{cohort}}/{{age_band}}/")

### Dashboard Integration

To integrate Cohort PGx network topology into the dashboard:

1. **Upload to S3** (in notebook 5 or deploy script):
   ```bash
   aws s3 sync 10_risk_dashboard/visualizations/cohort_pgx/ \
     s3://{{DASHBOARD_BUCKET}}/{{S3_PREFIX}}/cohort_pgx/ \
     --exclude "*.csv" --exclude "*.json" --include "*.html"
   ```

2. **Add API endpoint** (in Lambda `lambda_function.py`):
   ```python
   @app.get("/visualizations/cohort-pgx")
   def get_cohort_pgx_viz(cohort: str, age_band: str):
       age_band_fname = age_band.replace("-", "_")
       base_url = f"https://{DASHBOARD_BUCKET}/{S3_PREFIX}/cohort_pgx/{cohort}/{age_band_fname}"
       return {
           "network_topology": f"{base_url}/network_topology.html",
           "network_nodes": f"{base_url}/network_nodes.csv",
           "network_edges": f"{base_url}/network_edges.csv",
           "network_stats": f"{base_url}/network_stats.json"
       }
   ```

3. **Add dashboard tab** (in frontend `index.html`):
   - Create new tab "Cohort PGx" after "PGx Card"
   - Load network topology HTML via iframe from API endpoint
   - Show network statistics in sidebar

See `10_risk_dashboard/docs/` for full integration guide.

## API (reference)

Lambda receives **user input** (cohort, age_band, model/feature selections) and **filters** only—it does not process or generate visualization data. Scenario, BupaR, DTW, and FP-Growth visuals are **prebuilt on EC2** and **saved to S3**; the API returns **URLs or compact JSON payloads** for those prebuilt assets (filtered by cohort/age_band). Endpoints: `GET /visualizations/scenario`, `/visualizations/bupar`, `/visualizations/dtw`, `/visualizations/fpgrowth`, plus scenario importance/profile endpoints where available. See `10_risk_dashboard/backend/README.md`.

In [ ]:
print("Dashboard endpoints: 10_risk_dashboard/backend/README.md")
print("API Gateway deploy: utility_scripts/create_api_gateway_pgx_risk_calculator.sh")

## Next: Build and deploy

Build and deploy run **only** in [5_build_and_deploy.ipynb](5_build_and_deploy.ipynb). Run that notebook after this one.

**Run the cell below** to print all dashboard visual objects (file paths, S3 path, visual name, dashboard tab) and write `10_risk_dashboard/visualizations/dashboard_visual_objects.json` for notebook 5 to consume.

In [ ]:
# Print all dashboard visual objects (file paths + visual name / dashboard tab) for notebook 5 to consume
import json
import os
from pathlib import Path

# Paths (same as used in this notebook; define if not set by earlier cells)
_root = REPO_ROOT if 'REPO_ROOT' in dir() else Path.cwd()
_vis = _root / "10_risk_dashboard" / "visualizations"
_out = _root / "10_risk_dashboard" / "outputs"
_fi_base = _root / "3a_feature_importance" / "outputs"
_s3_prefix = os.environ.get("S3_DASHBOARD_PREFIX", "vcu/pgx-risk-calculator").strip("/") if 'os' in dir() else "vcu/pgx-risk-calculator"

dashboard_visual_objects = [
    # Feature Importance tab
    {"visual_name": "Feature importance heatmap (per cohort)", "dashboard_tab": "Feature Importance", "path": str(_fi_base / "{cohort}" / "plots" / "{cohort}_aggregated_fi_heatmap.png"), "path_type": "file_pattern", "s3_path": _s3_prefix + "/visualizations/feature_importance/{cohort}/aggregated_fi_heatmap.png", "notes": "per cohort; notebook 5 uploads to visualizations/feature_importance/{cohort}/"},
    {"visual_name": "Feature importance heatmap (combined)", "dashboard_tab": "Feature Importance", "path": str(_fi_base / "plots" / "combined_cohorts_feature_importance_heatmap.png"), "path_type": "file", "s3_path": _s3_prefix + "/visualizations/feature_importance/combined_cohorts_feature_importance_heatmap.png", "notes": "combined cohorts"},
    # BupaR tab
    {"visual_name": "BupaR process matrix and activity frequency", "dashboard_tab": "BupaR Process Mining", "path": str(_vis / "bupar"), "path_type": "directory", "s3_path": _s3_prefix + "/visualizations/bupar/{cohort}/{age_band}/plots/", "notes": "notebook 4 saves to local; notebook 5 Step 6 syncs to S3 final"},
    # DTW tab
    {"visual_name": "DTW chart_data.json + sequence_heatmap + plots", "dashboard_tab": "DTW Trajectories", "path": str(_vis / "dtw"), "path_type": "directory", "s3_path": _s3_prefix + "/visualizations/dtw/{cohort}/{age_band}/", "notes": "notebook 4 saves to local; notebook 5 Step 6 syncs to S3 final"},
    # FP-Growth tab
    {"visual_name": "FP-Growth itemsets and drug network", "dashboard_tab": "FP-Growth Patterns", "path": str(_vis / "fpgrowth"), "path_type": "directory", "s3_path": _s3_prefix + "/visualizations/fpgrowth/{cohort}/{age_band}/", "notes": "{cohort}/{age_band_fname}/: drug_name_*.json at root, plots/*.html, *.png; Step 6 sync", "static_files": ["drug_name_itemsets.json", "plots/{base}_combined_rules_network.html", "plots/{base}_drug_name_combined_top_itemsets.png"]},
    # PGx Cohort tab
    {"visual_name": "Cohort PGx network topology", "dashboard_tab": "PGx Cohort", "path": str(_vis / "cohort_pgx" / "networks"), "path_type": "directory", "s3_path": _s3_prefix + "/visualizations/cohort_pgx/networks/{cohort}/{age_band}/", "notes": "notebook 4 saves to local; notebook 5 Step 6 syncs to S3 final (sync_cohort_pgx_to_s3)"},
    # Scenario Analysis tab
    {"visual_name": "Scenario dashboard JSON", "dashboard_tab": "Scenario Analysis", "path": str(_vis / "causal" / "{cohort}" / "{age_band_fname}" / "dashboard_data.json"), "path_type": "file_pattern", "s3_path": _s3_prefix + "/visualizations/scenario/{cohort}/{age_band}/scenario_data.json", "notes": "upload_scenario_outputs_to_s3 → visualizations/scenario/ (S3 hyphen)"},
]

# Write JSON for notebook 5
_vis.mkdir(parents=True, exist_ok=True)
manifest_path = _vis / "dashboard_visual_objects.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump({"repo_root": str(_root), "visual_objects": dashboard_visual_objects}, f, indent=2)
print(f"Wrote manifest for notebook 5: {manifest_path}")
print()

# Print table: visual name | dashboard tab | path
print("Dashboard visual objects (for notebook 5):")
print("=" * 100)
for i, obj in enumerate(dashboard_visual_objects, 1):
    print(f"{i}. {obj['visual_name']}")
    print(f"   Tab:     {obj['dashboard_tab']}")
    print(f"   Path:    {obj['path']}")
    if obj.get('s3_path'):
        print(f"   S3 path: {obj['s3_path']}")
    print(f"   Type:    {obj['path_type']}")
    if obj.get("notes"):
        print(f"   Notes:   {obj['notes']}")
    print()
print("=" * 100)
print(f"Manifest: {manifest_path}")
print("Notebook 5 can load: json.load(open('10_risk_dashboard/visualizations/dashboard_visual_objects.json'))['visual_objects']")

In [ ]:
# Generate and save time-between-events histogram JSON for each cohort/age band
import pandas as pd
import numpy as np
import json
from pathlib import Path

# Set paths and cohorts/age bands as in notebook setup
FEATURE_ENGINEERING_DIR = Path('10_risk_dashboard/visualizations/dtw/feature_engineering')
HISTOGRAM_OUTPUT_DIR = Path('10_risk_dashboard/visualizations/dtw/outputs')

def save_time_between_events_histogram(cohort, age_band, bins=30):
    age_band_fname = age_band.replace('-', '_')
    parquet_path = FEATURE_ENGINEERING_DIR / f'dtw_features_{cohort}_{age_band_fname}.parquet'
    output_dir = HISTOGRAM_OUTPUT_DIR / cohort / age_band_fname
    output_dir.mkdir(parents=True, exist_ok=True)
    json_path = output_dir / 'time_between_events_histogram.json'

    if not parquet_path.exists():
        print(f'File not found: {parquet_path}')
        return

    df = pd.read_parquet(parquet_path)
    values = df['mean_days_between_events'].dropna().values
    if len(values) == 0:
        print(f'No valid mean_days_between_events for {cohort} {age_band}')
        return

    counts, bin_edges = np.histogram(values, bins=bins)
    histogram_json = {
        "type": "histogram",
        "x": values.tolist(),
        "nbinsx": bins,
        "name": "Time Between Events",
        "x_label": "Mean Days Between Events",
        "y_label": "Patient Count",
        "counts": counts.tolist(),
        "bin_edges": bin_edges.tolist()
    }
    with open(json_path, 'w') as f:
        json.dump(histogram_json, f, indent=2)
    print(f'Saved histogram JSON: {json_path}')

# Example usage for one cohort/age band
save_time_between_events_histogram('opioid_ed', '65-74')

## Manuscript Checkpoint Writer

Write structured JSON checkpoints to `s3://pgxdatalake/gold/manuscript_checkpoints/` for all
visualization outputs needed to fill manuscript placeholders.  Reads from **local** outputs
produced earlier in this notebook (FP-Growth rules JSON, DTW chart_data.json, SHAP parquet,
PGx coverage CSV).  Run after all visualization cells complete.

| Output | S3 key | Manuscript use |
|--------|--------|----------------|
| `fpgrowth_manuscript_summary.json` | `gold/manuscript_checkpoints/fpgrowth/{cohort}/{ab}/` | Top drug-pair rules (confidence, lift, support) |
| `dtw_manuscript_summary.json` | `gold/manuscript_checkpoints/dtw/{cohort}/{ab}/` | Trajectory N, cluster N, target split |
| `shap_manuscript_summary.json` | `gold/manuscript_checkpoints/shap/{cohort}/{ab}/` | Top-10 features by mean |SHAP| |
| `pgx_manuscript_summary.json` | `gold/manuscript_checkpoints/pgx/{cohort}/{ab}/` | PGx feature coverage % |

In [ ]:

# ── Manuscript Checkpoint Writer — ALL cohorts / ALL age bands / ALL bins ─────
# Writes structured JSON to s3://pgxdatalake/gold/manuscript_checkpoints/
# Uses REQUIRED_COHORTS (all age bands including 0-12) not DASHBOARD_VISUAL_AGE_BANDS.
# Iterates all 4 density bins (low/medium/high/extreme) per cohort/age_band.

import boto3, json, traceback
import pandas as pd
import numpy as np
from pathlib import Path
from py_helpers.event_density_utils import DENSITY_BINS  # ('low','medium','high','extreme')

try:
    from py_helpers.constants import REQUIRED_COHORTS
except ImportError:
    _all_bands = ['0-12','13-24','25-44','45-54','55-64','65-74','75-84','85-114']
    REQUIRED_COHORTS = {"opioid_ed": _all_bands, "non_opioid_ed": _all_bands}

s3 = boto3.client("s3", region_name="us-east-1")
CHKPT_PREFIX = "gold/manuscript_checkpoints"
CHKPT_BUCKET = S3_BUCKET  # pgxdatalake

BUPAR_VISUALS    = VISUAL_ROOT / "bupar"
FPGROWTH_VISUALS = VISUAL_ROOT / "fpgrowth"
DTW_VISUALS      = VISUAL_ROOT / "dtw"
COHORT_PGX_NETWORKS = VISUAL_ROOT / "cohort_pgx" / "networks"
SHAP_OUTPUTS     = REPO_ROOT / "7_shap_analysis" / "outputs"
FFA_OUTPUTS      = REPO_ROOT / "8_ffa_analysis" / "outputs"
PGX_CHECKPOINT   = REPO_ROOT / "5_pgx_analysis" / "outputs" / "feature_engineering"

def _put_json(key: str, obj: dict) -> None:
    s3.put_object(
        Bucket=CHKPT_BUCKET, Key=key,
        Body=json.dumps(obj, indent=2, default=str).encode(),
        ContentType="application/json",
    )

def _bupar_summary_from_activity_freq(json_path: Path) -> dict:
    """Extract n_patients, n_activities, top_activity from activity_frequency.json."""
    if not json_path.exists():
        return {"note": f"not found: {json_path.name}"}
    try:
        af = json.loads(json_path.read_text(encoding="utf-8"))
    except Exception as e:
        return {"note": f"parse error: {e}"}
    data = af.get("data", {})
    # Handle R-written format: "data" is a list of row dicts {activity, count, year, ...}
    if isinstance(data, list):
        from collections import defaultdict
        _totals: dict = defaultdict(int)
        for row in data:
            if isinstance(row, dict):
                code = str(row.get("activity", row.get("activity_short", "")))
                _totals[code] += int(row.get("count", 0) or 0)
        data = dict(_totals)
    n_activities = len(data)
    top_activity = None
    if data:
        totals = {k: sum(v) if isinstance(v, list) else v for k, v in data.items()}
        top_activity = max(totals, key=totals.get)
    return {
        "n_patients":   af.get("n_patients", 0),
        "n_activities": n_activities,
        "top_activity": top_activity,
        "year_labels":  af.get("year_labels", []),
        "empty":        af.get("empty", n_activities == 0),
    }

def _top_rules_from_dir(fpg_dir: Path, n: int = 10):
    """Read all *rules*.json under fpg_dir, return top-n rules by confidence."""
    rule_files = sorted(fpg_dir.rglob("*rules_target_only.json")) or \
                 sorted(fpg_dir.rglob("*rules.json"))
    all_rules = []
    total = 0
    for rf in rule_files:
        try:
            rules = json.loads(rf.read_text(encoding="utf-8"))
            if not isinstance(rules, list):
                continue
            total += len(rules)
            for r in rules:
                all_rules.append({
                    "antecedents": r.get("antecedents", r.get("items", "")),
                    "consequents":  r.get("consequents", ""),
                    "support":     round(float(r.get("support",    0)), 4),
                    "confidence":  round(float(r.get("confidence", 0)), 4),
                    "lift":        round(float(r.get("lift",       0)), 4),
                    "source_file": rf.name,
                })
        except Exception:
            pass
    seen, dedup = set(), []
    for r in sorted(all_rules, key=lambda x: x["confidence"], reverse=True):
        k = str(r["antecedents"]) + str(r["consequents"])
        if k not in seen:
            seen.add(k); dedup.append(r)
        if len(dedup) >= n:
            break
    return total, dedup

def _dtw_summary_from_chart(chart_path: Path) -> dict:
    """Extract trajectory summary + per-archetype breakdown from chart_data.json."""
    if not chart_path.exists():
        return {"note": "chart_data.json not found"}
    cd = json.loads(chart_path.read_text(encoding="utf-8"))
    sb = cd.get("summary", cd)
    out = {
        "total_trajectories": sb.get("total_trajectories", 0),
        "trajectory_length":  sb.get("trajectory_length", {}),
        "target_counts":      sb.get("target_counts", {}),
        "n_clusters":         len(cd.get("clusters", cd.get("cluster_labels", []))),
        "routine_vs_no_routine": cd.get("routine_comparison", {}),
    }
    hr = cd.get("high_risk_trajectories", {})
    if hr and isinstance(hr, dict) and "x" in hr and "n" in hr:
        x_labels, n_vals, y_vals = hr["x"], hr["n"], hr["y"]
        total_n = sum(n_vals) if n_vals else 0
        archetypes = [
            {"archetype": label, "n": int(n_val),
             "pct_of_total": round(int(n_val) / total_n * 100, 1) if total_n else 0,
             "target_rate": round(float(y_val), 4)}
            for label, n_val, y_val in zip(x_labels, n_vals, y_vals)
        ]
        out["archetypes_by_dtw_quartile"] = archetypes
        if len(archetypes) >= 4:
            out["rapid_onset"]        = archetypes[0]
            out["chronic_escalation"] = archetypes[-1]
    ttt = cd.get("time_to_target_sequences", {})
    if ttt and isinstance(ttt, dict) and "x" in ttt and "y" in ttt:
        out["time_to_target_days"] = {
            "by_routine": [
                {"bucket": x, "mean_days": round(float(y), 1),
                 "mean_months": round(float(y) / 30.44, 1), "n": int(n)}
                for x, y, n in zip(ttt["x"], ttt["y"], ttt.get("n", [0]*len(ttt["y"])))
            ]
        }
    tpp = cd.get("target_pathway_patterns", {})
    if tpp and isinstance(tpp, dict):
        out["target_pathway_patterns"] = {
            "top_patterns": tpp.get("patterns", tpp.get("x", []))[:5],
            "pattern_frequencies": tpp.get("y", [])[:5],
        }
    return out

def _shap_top10_from_dir(shap_dir: Path) -> list:
    """Return top-10 SHAP features from *shap_global_importance*.csv (prefer xgboost)."""
    if not shap_dir.exists():
        return []
    csv_files = (
        sorted(shap_dir.rglob("*shap_global_importance_xgboost.csv"))
        or sorted(shap_dir.rglob("*shap_global_importance_catboost.csv"))
        or sorted(shap_dir.rglob("*shap_global_importance*.csv"))
    )
    if not csv_files:
        return []
    dfs = []
    for csv_path in csv_files:
        try:
            df = pd.read_csv(csv_path)
            if "feature" in df.columns and "mean_abs_shap" in df.columns:
                dfs.append(df[["feature", "mean_abs_shap"]])
        except Exception:
            pass
    if not dfs:
        return []
    combined = (
        pd.concat(dfs, ignore_index=True)
        .groupby("feature", as_index=False)["mean_abs_shap"]
        .mean()
        .sort_values("mean_abs_shap", ascending=False)
        .head(10)
        .reset_index(drop=True)
    )
    return [
        {"rank": i + 1, "feature": row["feature"],
         "mean_abs_shap": round(float(row["mean_abs_shap"]), 6)}
        for i, row in combined.iterrows()
    ]

def _pgx_net_html(networks_base: Path, cohort_name: str, age_band_fname: str,
                  bin_name: str | None = None) -> Path:
    """Resolve network_topology.html path for full-cohort or per-bin."""
    if bin_name:
        return networks_base / cohort_name / age_band_fname / "density" / bin_name / "network_topology.html"
    return networks_base / cohort_name / age_band_fname / "network_topology.html"

written = errors = 0
all_cohorts_bands = [
    (cohort, ab)
    for cohort, bands in REQUIRED_COHORTS.items()
    for ab in bands
]

for cohort_name, age_band in all_cohorts_bands:
    age_band_fname = age_band.replace("-", "_")
    print(f"\n── {cohort_name}/{age_band} ──")

    bupar_base = BUPAR_VISUALS    / cohort_name / age_band_fname
    fpg_base   = FPGROWTH_VISUALS / cohort_name / age_band_fname
    dtw_base   = DTW_VISUALS      / cohort_name / age_band_fname
    shap_base  = SHAP_OUTPUTS     / cohort_name / age_band_fname
    ffa_base   = FFA_OUTPUTS      / cohort_name / age_band_fname
    base       = f"{cohort_name}_{age_band_fname}"

    # ── PGx coverage — cohort/age_band level only ─────────────────────────────
    try:
        pgx_csv = PGX_CHECKPOINT / f"pgx_added_features_{cohort_name}_{age_band_fname}.csv"
        pgx_summary = {"cohort": cohort_name, "age_band": age_band}
        if pgx_csv.exists():
            pgx_df = pd.read_csv(pgx_csv)
            n_patients = len(pgx_df)
            if "pgx_num_drugs" in pgx_df.columns:
                n_pgx = int((pgx_df["pgx_num_drugs"] > 0).sum())
                pgx_summary.update({
                    "n_patients":      n_patients,
                    "n_with_pgx":      n_pgx,
                    "pct_coverage":    round(n_pgx / n_patients * 100, 2) if n_patients else 0,
                    "mean_pgx_drugs":  round(float(pgx_df["pgx_num_drugs"].mean()), 3),
                    "mean_cpic_drugs": round(float(pgx_df["pgx_num_cpic_drugs"].mean()), 3)
                                       if "pgx_num_cpic_drugs" in pgx_df.columns else None,
                })
            else:
                pgx_summary["pgx_columns"] = [c for c in pgx_df.columns if c.startswith("pgx_")]
                pgx_summary["n_patients"]  = n_patients
        else:
            pgx_summary["note"] = f"not found: {pgx_csv.name}"
        key = f"{CHKPT_PREFIX}/pgx/{cohort_name}/{age_band}/pgx_manuscript_summary.json"
        _put_json(key, pgx_summary)
        pct = pgx_summary.get("pct_coverage", "?")
        print(f"  [PGx] coverage={pct}%  ✓")
        written += 1
    except Exception as e:
        print(f"  [PGx] ✗ {e}")
        errors += 1

    # ── Cohort PGx full-cohort network ────────────────────────────────────────
    try:
        net_html = _pgx_net_html(COHORT_PGX_NETWORKS, cohort_name, age_band_fname)
        net_summary = {
            "cohort": cohort_name, "age_band": age_band,
            "network_html_exists": net_html.exists(),
            "network_html_path":   str(net_html.relative_to(REPO_ROOT)) if net_html.exists() else "not found",
        }
        key = f"{CHKPT_PREFIX}/cohort_pgx/{cohort_name}/{age_band}/cohort_pgx_manuscript_summary.json"
        _put_json(key, net_summary)
        print(f"  [CohortPGx] network_html={'✓' if net_html.exists() else '✗ not found'}")
        written += 1
    except Exception as e:
        print(f"  [CohortPGx] ✗ {e}")
        errors += 1

    # ── Per-bin loop — BupaR / FP-Growth / DTW / CohortPGx / SHAP / FFA ──────
    for bin_name in DENSITY_BINS:  # low, medium, high, extreme

        # ── BupaR — per-bin: density/{bin}/plots/{base}_activity_frequency.json ─
        try:
            bin_bupar_json = bupar_base / "density" / bin_name / "plots" / f"{base}_activity_frequency.json"
            if not bin_bupar_json.exists():
                bin_bupar_json = bupar_base / "plots" / f"{base}_activity_frequency.json"
            bupar_data = _bupar_summary_from_activity_freq(bin_bupar_json)
            bupar_summary = {
                "cohort": cohort_name, "age_band": age_band, "bin": bin_name,
                **bupar_data,
                "source_file": str(bin_bupar_json.relative_to(REPO_ROOT)) if bin_bupar_json.exists() else "not found",
            }
            key = f"{CHKPT_PREFIX}/bupar/{cohort_name}/{age_band}/{bin_name}/bupar_manuscript_summary.json"
            _put_json(key, bupar_summary)
            top_act = bupar_data.get("top_activity", "n/a")
            print(f"  [BupaR/{bin_name}] n_activities={bupar_data.get('n_activities', 0)}  top={top_act}  ✓")
            written += 1
        except Exception as e:
            print(f"  [BupaR/{bin_name}] ✗ {e}")
            errors += 1

        # ── FP-Growth — per-bin: density/{bin}/ ───────────────────────────────
        try:
            bin_fpg = fpg_base / "density" / bin_name
            search_dir = bin_fpg if bin_fpg.exists() else fpg_base
            total_rules, top_rules = _top_rules_from_dir(search_dir)
            fpg_summary = {
                "cohort": cohort_name, "age_band": age_band, "bin": bin_name,
                "total_rules": total_rules,
                "top_rules":   top_rules,
                "source_dir":  str(search_dir.relative_to(REPO_ROOT)),
            }
            key = f"{CHKPT_PREFIX}/fpgrowth/{cohort_name}/{age_band}/{bin_name}/fpgrowth_manuscript_summary.json"
            _put_json(key, fpg_summary)
            print(f"  [FPG/{bin_name}] total_rules={total_rules}  ✓")
            written += 1
        except Exception as e:
            print(f"  [FPG/{bin_name}] ✗ {e}")
            errors += 1

        # ── DTW — per-bin: density/{bin}/chart_data.json ──────────────────────
        try:
            bin_dtw = dtw_base / "density" / bin_name
            chart_path = (bin_dtw / "chart_data.json") if bin_dtw.exists() \
                         else (dtw_base / "chart_data.json")
            dtw_data = _dtw_summary_from_chart(chart_path)
            dtw_summary = {"cohort": cohort_name, "age_band": age_band, "bin": bin_name, **dtw_data}
            key = f"{CHKPT_PREFIX}/dtw/{cohort_name}/{age_band}/{bin_name}/dtw_manuscript_summary.json"
            _put_json(key, dtw_summary)
            n_traj = dtw_data.get("total_trajectories", 0)
            ro_n = dtw_data.get("rapid_onset", {}).get("n", "?")
            ce_n = dtw_data.get("chronic_escalation", {}).get("n", "?")
            print(f"  [DTW/{bin_name}] trajectories={n_traj}  rapid_onset_n={ro_n}  chronic_escalation_n={ce_n}  ✓")
            written += 1
        except Exception as e:
            print(f"  [DTW/{bin_name}] ✗ {e}")
            errors += 1

        # ── Cohort PGx — per-bin: density/{bin}/network_topology.html ─────────
        try:
            bin_net_html = _pgx_net_html(COHORT_PGX_NETWORKS, cohort_name, age_band_fname, bin_name)
            pgx_net_summary = {
                "cohort": cohort_name, "age_band": age_band, "bin": bin_name,
                "network_html_exists": bin_net_html.exists(),
                "network_html_path":   str(bin_net_html.relative_to(REPO_ROOT)) if bin_net_html.exists() else "not found",
            }
            key = f"{CHKPT_PREFIX}/cohort_pgx/{cohort_name}/{age_band}/{bin_name}/cohort_pgx_manuscript_summary.json"
            _put_json(key, pgx_net_summary)
            print(f"  [CohortPGx/{bin_name}] network_html={'✓' if bin_net_html.exists() else '✗ not found'}")
            written += 1
        except Exception as e:
            print(f"  [CohortPGx/{bin_name}] ✗ {e}")
            errors += 1

        # ── SHAP top-10 — per-bin: bin_models/{bin}/ ──────────────────────────
        try:
            bin_shap = shap_base / "bin_models" / bin_name
            search_shap = bin_shap if bin_shap.exists() else shap_base
            top10 = _shap_top10_from_dir(search_shap)
            shap_summary = {
                "cohort": cohort_name, "age_band": age_band, "bin": bin_name,
                "top_features": top10,
                "source_dir": str(search_shap.relative_to(REPO_ROOT)) if search_shap.exists() else "not found",
            }
            key = f"{CHKPT_PREFIX}/shap/{cohort_name}/{age_band}/{bin_name}/shap_manuscript_summary.json"
            _put_json(key, shap_summary)
            top1 = top10[0]["feature"] if top10 else "n/a"
            print(f"  [SHAP/{bin_name}] top={top1}  ✓")
            written += 1
        except Exception as e:
            print(f"  [SHAP/{bin_name}] ✗ {e}")
            errors += 1

        # ── FFA causal — per-bin: bin_models/{bin}/ ───────────────────────────
        try:
            bin_ffa = ffa_base / "bin_models" / bin_name
            causal_csv = bin_ffa / "ffa_causal_factors.csv"
            if not causal_csv.exists():
                causal_csv = ffa_base / "ffa_causal_factors.csv"
            ffa_summary = {"cohort": cohort_name, "age_band": age_band, "bin": bin_name,
                           "n_causal_features": 0, "top_features": []}
            if causal_csv.exists():
                cf = pd.read_csv(causal_csv)
                ffa_summary["n_causal_features"] = len(cf)
                ir_col   = next((c for c in ("intervention_rate","ir","IR","importance",
                                             "ffa_importance","mean_abs_shap") if c in cf.columns), None)
                feat_col = next((c for c in ("feature","feature_name","code","item") if c in cf.columns), None)
                if feat_col and ir_col:
                    top = cf.nlargest(15, ir_col)[[feat_col, ir_col]].copy()
                    top.columns = ["feature", "ir_score"]
                    ffa_summary["top_features"] = top.to_dict(orient="records")
            key = f"{CHKPT_PREFIX}/ffa/{cohort_name}/{age_band}/{bin_name}/ffa_manuscript_summary.json"
            _put_json(key, ffa_summary)
            print(f"  [FFA/{bin_name}] features={ffa_summary['n_causal_features']}  ✓")
            written += 1
        except Exception as e:
            print(f"  [FFA/{bin_name}] ✗ {e}")
            errors += 1

print(f"\n{'='*70}")
print(f"Manuscript checkpoints written: {written}  errors: {errors}")
print(f"Cohorts: {list(REQUIRED_COHORTS.keys())}")
print(f"Age bands: all (including 0-12)")
print(f"Bins: {list(DENSITY_BINS)}")
print(f"S3: s3://{CHKPT_BUCKET}/{CHKPT_PREFIX}/")
print()
print("Run locally after notebooks complete:")
print("  python manuscript/scripts/extract_visual_manuscript.py")
